In [1]:
# Import packages and declare control variables

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from openpyxl import load_workbook

In [2]:
# Declare paths, files, and load data

path_TTpaper_SI = 'C:/Users/skar/repos/US_WWTP_GHG/supplementary_databases'
fi_SI_A = 'supplementary_database_A.xlsx'
fi_SI_B = 'supplementary_database_B.xlsx'
fi_SI_C = 'supplementary_database_C.xlsx'
path_TTpaper_energy = 'C:/Users/skar/Box/_Saura_Self/Proj - Water tool analysis/literature/Abbadi et al 2025/Energy_data'
fi_energy = 'facility_info_energy_demand.xlsx'
path_TTpaper_chemicals = 'C:/Users/skar/Box/_Saura_Self/Proj - Water tool analysis/literature/Abbadi et al 2025/Chemicals_data'
fi_chemicals = 'ANL_chemical_data.xlsx'

path_data = 'C:/Users/skar/Box/FY25_Water Analysis_FCG share/3. Survey data collection/2025 Project (CURRENT)'
fi_data = 'Official_Top_50_Plants_Data_Repository.xlsx'
fi_select_plants_sheet = 'Db'
fi_all_plants = 'all_wwtps_data_070124'

path_DMR = 'C:/Users/skar/Box/_Saura_Self/Proj - Water tool analysis/data/DMR data_dwd_12192025'

path_data_2 = 'C:/Users/skar/Box/_Saura_Self/Proj - Water tool analysis/data'
path_out = 'C:/Users/skar/Box/_Saura_Self/Proj - Water tool analysis/data/output'
path_out_plots = 'C:/Users/skar/Box/_Saura_Self/Proj - Water tool analysis/plots'
fi_out = 'survey_data_repo.xlsx'

df_SI_A = pd.read_excel(path_TTpaper_SI + '/' + fi_SI_A, header=0)
df_SI_B = pd.read_excel(path_TTpaper_SI + '/' + fi_SI_B, header=0)
df_SI_C = pd.read_excel(path_TTpaper_SI + '/' + fi_SI_C, header=0)
df_energy = pd.read_excel(path_TTpaper_energy + '/' + fi_energy, header=2)
df_chemicals = pd.read_excel(path_TTpaper_chemicals + '/' + fi_chemicals, header=0)

df_select_plants = pd.read_excel(path_data + '/' + fi_data, fi_select_plants_sheet, header=0)
#df_all_plants = pd.read_excel(path_data + '/' + fi_data, fi_all_plants, header=0)

write_to_excel_tab = True


In [3]:
# View tables

print ("SI Table A:\n", df_SI_A.head())
print ("SI Table B:\n", df_SI_B.head())
print ("SI Table C:\n", df_SI_C.head())
print ("Energy Data:\n", df_energy.head())
print ("Chemicals Data:\n", df_chemicals.head())
print ("Selected Plants, columns:\n", list(df_select_plants.columns))
#print ("All Plants Data:\n", df_all_plants.head())

SI Table A:
    FACILITY_ID     CWNS_NUM  AED  AND  AS  AS-A2O  AS-BDENIT  AS-EA  AS-OD  \
0      1159112  30000044001    0    0   0       0          0      0      0   
1      1159221  30000180001    0    1   0       0          0      0      0   
2      1160522  28001370001    0    0   0       0          0      0      0   
3      1164458  41000244001    1    0   1       0          0      0      0   
4      1165962  47000010001    0    0   0       1          0      0      0   

   AS-P  ...  N2  O1  O1E  O2  O3  O5  O6  TT_IDENTIFIED  \
0     0  ...   0   0    0   0   0   0   0              1   
1     0  ...   0   0    0   0   0   0   0              1   
2     0  ...   0   0    0   0   0   0   0              1   
3     0  ...   0   0    0   0   0   0   0              1   
4     0  ...   0   0    0   0   0   0   0              1   

                              TT_ASSIGN_NOTE  UP_ID_NOTE  
0  Assigned based on reported unit processes         NaN  
1  Assigned based on reported unit proc

In [4]:
# Data clean up and save

# Dictionary to store all DataFrames
all_data = {}

# Filter completed data set
df_select = df_select_plants[df_select_plants['Study Status'] == 'Complete'].copy()
# remove columns with all NaN values
df_select = df_select.dropna(axis=1, how='all')
# remove rows with all NaN values
df_select = df_select.dropna(axis=0, how='all')
# reset index
df_select = df_select.reset_index(drop=True)
# Standardize column names by stripping leading/trailing spaces
df_select.columns = df_select.columns.str.strip()
# Standardize string entries by stripping leading/trailing spaces
df_select = df_select.map(lambda x: x.strip() if isinstance(x, str) else x)
# Standardize column names by replacing spaces with underscores
df_select.columns = df_select.columns.str.replace(' ', '_')
# Standardize column names by replacing slashes with underscores
df_select.columns = df_select.columns.str.replace('/', '_')
# Standardize column names by replacing hyphens with underscores
df_select.columns = df_select.columns.str.replace('-', '_')

df_select.drop(columns=['Study_Status',                         
                        'FOIA_STATUS_(Not_Requested,_Requested,_Filled,_Denied,_Other)', 
                        'PRIMARY_CONTACT_NAME',
                        'PRIMARY_CONTACT_EMAIL', 
                        'RESPONSE_STATUS',
                        'DATA_URL',
                        'PROVIDED_TT_DATA',
                        'REQUEST_CODE',                        
                        'OTHER_NOTES'
], inplace=True)

# Function to finalize DataFrame before adding to all_data
def finalize_dataframe(df, df_name):
    """
    Finalize DataFrame by removing duplicates and ensuring data quality
    
    Parameters:
    - df: DataFrame to finalize
    - df_name: Name of the DataFrame for logging
    
    Returns:
    - Cleaned DataFrame
    """
    print(f"Finalizing '{df_name}' DataFrame...")
    
    # Check initial state
    initial_rows = len(df)
    initial_cols = len(df.columns)
    print(f"  Initial: {initial_rows} rows, {initial_cols} columns")
    
    # Remove duplicate rows based on CWNS_ID (keep first occurrence)
    df_clean = df.drop_duplicates(subset=['CWNS_ID'], keep='first')
    
    # Check for duplicate removal
    if len(df_clean) != initial_rows:
        removed_rows = initial_rows - len(df_clean)
        print(f"  Removed {removed_rows} duplicate CWNS_ID rows")
    
    # Remove duplicate columns (keep first occurrence)
    df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()]
    
    # Check for duplicate column removal
    if len(df_clean.columns) != initial_cols:
        removed_cols = initial_cols - len(df_clean.columns)
        print(f"  Removed {removed_cols} duplicate columns")
    
    # Reset index
    df_clean = df_clean.reset_index(drop=True)
    
    # Final state
    final_rows = len(df_clean)
    final_cols = len(df_clean.columns)
    print(f"  Final: {final_rows} rows, {final_cols} columns")
    
    # Validate CWNS_ID uniqueness
    unique_cwns = df_clean['CWNS_ID'].nunique()
    if unique_cwns != final_rows:
        print(f"  WARNING: CWNS_ID not unique! {unique_cwns} unique vs {final_rows} rows")
    else:
        print(f"  ✓ All CWNS_IDs are unique")
    
    return df_clean

"""
# Check treatment train assignment data availability
df_sub = df_select[[
    'CWNS_ID',
    'PREDICTED_WRRF_TT_CODE']].copy()
df_sub['CWNS_ID'] = df_select['CWNS_ID'].astype(int)
print("Rows with no treatment train assignment:")
print(df_sub[df_sub['PREDICTED_WRRF_TT_CODE'].isna()])
# remove rows with no treatment train assignment
df_sub = df_sub[~df_sub['PREDICTED_WRRF_TT_CODE'].isna()].reset_index(drop=True)
# Finalize and add to all_data
all_data['Treatment_Train'] = finalize_dataframe(df_sub, 'Treatment_Train')
"""

# Incorporate TT identification, codification, and comparison of TTs with Abby et al., 2025
# read updated predicted and assigned TTs identified following Abby et al., 2025
df_TTs = pd.read_excel(os.path.join(path_data, 'TT_ds_results_with_formula.xlsx'), sheet_name='TT_ds_results', usecols=['CWNS_ID', 'PREDICTED_WRRF_TT_CODE', 'TT_IDENTIFIED', 'Assigned TT'])
print(df_TTs.head())
# Drop current TTs info column
#df_select.drop(columns=['PREDICTED_WRRF_TT_CODE'], inplace=True)
#df_select = df_select.merge(df_TTs, on=['CWNS_ID'], how='left')
# Finalize and add to all_data
all_data['TTs'] = finalize_dataframe(df_TTs, 'TTs')

# Check flow data availability
df_sub = df_select[[
'CWNS_ID',
'FACILITY_ID',
'FACILITY_NAME',
'LATITUDE',
'LONGITUDE',
'CITY',
'STATE_CODE',
'AUTHORITY_NAME',
'COUNTY_NAME',

'DESIGN_FLOW',
'DESIGN_FLOW_UNITS',
'ACTUAL_FLOW_MIN',
'ACTUAL_FLOW_MAX',
'ACTUAL_FLOW_AVG',
'ACTUAL_FLOW_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)

print("Rows with no flow data:")
print(df_sub[df_sub['DESIGN_FLOW'].isna() & 
                    df_sub['ACTUAL_FLOW_MIN'].isna() & 
                    df_sub['ACTUAL_FLOW_MAX'].isna() & 
                    df_sub['ACTUAL_FLOW_AVG'].isna()])

# remove rows with no flow data
df_sub = df_sub[~(df_sub['DESIGN_FLOW'].isna() & 
                                df_sub['ACTUAL_FLOW_MIN'].isna() & 
                                df_sub['ACTUAL_FLOW_MAX'].isna() & 
                                df_sub['ACTUAL_FLOW_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['Flow'] = finalize_dataframe(df_sub, 'Flow')

# Check BOD5 data availability
df_sub = df_select[[
    'CWNS_ID',
    'BOD5_MIN',
    'BOD5_MAX',
    'BOD5_AVG',
    'BOD5_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no BOD5 data:")
print(df_sub[df_sub['BOD5_MIN'].isna() & 
                    df_sub['BOD5_MAX'].isna() & 
                    df_sub['BOD5_AVG'].isna()])
# remove rows with no BOD5 data
df_sub = df_sub[~(df_sub['BOD5_MIN'].isna() & 
                                df_sub['BOD5_MAX'].isna() & 
                                df_sub['BOD5_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['BOD5'] = finalize_dataframe(df_sub, 'BOD5')

# Check CBOD5 data availability
df_sub = df_select[[
    'CWNS_ID',
    'CBOD5_MIN',
    'CBOD5_MAX',
    'CBOD5_AVG',
    'CBOD5_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no CBOD5 data:")
print(df_sub[df_sub['CBOD5_MIN'].isna() & 
                    df_sub['CBOD5_MAX'].isna() & 
                    df_sub['CBOD5_AVG'].isna()])
# remove rows with no CBOD5 data
df_sub = df_sub[~(df_sub['CBOD5_MIN'].isna() & 
                               df_sub['CBOD5_MAX'].isna() & 
                                df_sub['CBOD5_AVG'].isna())].reset_index(drop=True)    
# Finalize and add to all_data
all_data['CBOD5'] = finalize_dataframe(df_sub, 'CBOD5')

# Check SS data availability
df_sub = df_select[[  
    'CWNS_ID',
    'SS_MIN',
    'SS_MAX',
    'SS_AVG',
    'SS_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no SS data:")
print(df_sub[df_sub['SS_MIN'].isna() & 
                    df_sub['SS_MAX'].isna() & 
                    df_sub['SS_AVG'].isna()])
# remove rows with no SS data
df_sub = df_sub[~(df_sub['SS_MIN'].isna() & 
                               df_sub['SS_MAX'].isna() &
                                df_sub['SS_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['SS'] = finalize_dataframe(df_sub, 'SS')

# Check VSS data availability
df_sub = df_select[[
    'CWNS_ID',
    'VSS_MIN',
    'VSS_MAX',
    'VSS_AVG',
    'VSS_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no VSS data:")
print(df_sub[df_sub['VSS_MIN'].isna() & 
                    df_sub['VSS_MAX'].isna() & 
                    df_sub['VSS_AVG'].isna()])
# remove rows with no VSS data
df_sub = df_sub[~(df_sub['VSS_MIN'].isna() & 
                               df_sub['VSS_MAX'].isna() & 
                                df_sub['VSS_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['VSS'] = finalize_dataframe(df_sub, 'VSS')

# Check TS data availability
df_sub = df_select[[
    'CWNS_ID',
    'TS_MIN',
    'TS_MAX',
    'TS_AVG',
    'TS_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no TS data:") 
print(df_sub[df_sub['TS_MIN'].isna() & 
                    df_sub['TS_MAX'].isna() & 
                    df_sub['TS_AVG'].isna()])
# remove rows with no TS data
df_sub = df_sub[~(df_sub['TS_MIN'].isna() & 
                               df_sub['TS_MAX'].isna() & 
                                df_sub['TS_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['TS'] = finalize_dataframe(df_sub, 'TS')

# Check TSS data availability
df_sub = df_select[[
    'CWNS_ID',
    'TSS_MIN',
    'TSS_MAX',
    'TSS_AVG',
    'TSS_UNITS']].copy()   
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no TSS data:")
print(df_sub[df_sub['TSS_MIN'].isna() &
                    df_sub['TSS_MAX'].isna() & 
                    df_sub['TSS_AVG'].isna()])
# remove rows with no TSS data
df_sub = df_sub[~(df_sub['TSS_MIN'].isna() & 
                               df_sub['TSS_MAX'].isna() & 
                                df_sub['TSS_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['TSS'] = finalize_dataframe(df_sub, 'TSS')

# Check VTS data availability
df_sub = df_select[[
    'CWNS_ID',
    'VTS_MIN',
    'VTS_MAX',
    'VTS_AVG',
    'VTS_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no VTS data:")
print(df_sub[df_sub['VTS_MIN'].isna() & 
                    df_sub['VTS_MAX'].isna() & 
                    df_sub['VTS_AVG'].isna()])
# remove rows with no VTS data
df_sub = df_sub[~(df_sub['VTS_MIN'].isna() & 
                               df_sub['VTS_MAX'].isna() & 
                                df_sub['VTS_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['VTS'] = finalize_dataframe(df_sub, 'VTS')

# Check TKN data availability
df_sub = df_select[[
    'CWNS_ID',
    'TKN_MIN',
    'TKN_MAX',
    'TKN_AVG',
    'TKN_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no TKN data:")
print(df_sub[df_sub['TKN_MIN'].isna() &
                    df_sub['TKN_MAX'].isna() & 
                    df_sub['TKN_AVG'].isna()])
# remove rows with no TKN data
df_sub = df_sub[~(df_sub['TKN_MIN'].isna() & 
                               df_sub['TKN_MAX'].isna() &
                                df_sub['TKN_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['TKN'] = finalize_dataframe(df_sub, 'TKN')

# Check TN data availability
df_sub = df_select[[
    'CWNS_ID',
    'TN_MIN',
    'TN_MAX',
    'TN_AVG',
    'TN_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no TN data:")
print(df_sub[df_sub['TN_MIN'].isna() &
                    df_sub['TN_MAX'].isna() & 
                    df_sub['TN_AVG'].isna()])
# remove rows with no TN data
df_sub = df_sub[~(df_sub['TN_MIN'].isna() & 
                               df_sub['TN_MAX'].isna() & 
                                df_sub['TN_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['TN'] = finalize_dataframe(df_sub, 'TN')

# Check NH3 data availability
df_sub = df_select[[
    'CWNS_ID',
    'NH3_MIN',
    'NH3_MAX',
    'NH3_AVG',
    'NH3_UNITS',
    'NH3_N_MIN',
    'NH3_N_MAX',
    'NH3_N_AVG',
    'NH3_N_UNITS',
    ]].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no NH3 data:")
print(df_sub[df_sub['NH3_MIN'].isna() &
                    df_sub['NH3_MAX'].isna() & 
                    df_sub['NH3_AVG'].isna() &
                    df_sub['NH3_N_MIN'].isna() &
                    df_sub['NH3_N_MAX'].isna() & 
                    df_sub['NH3_N_AVG'].isna()])
# remove rows with no NH3 data
df_sub = df_sub[~(df_sub['NH3_MIN'].isna() & 
                                df_sub['NH3_MAX'].isna() &
                                df_sub['NH3_AVG'].isna() &
                                df_sub['NH3_N_MIN'].isna() &
                                df_sub['NH3_N_MAX'].isna() &
                                df_sub['NH3_N_AVG'].isna())].reset_index(drop=True)
for col in ["NH3_N_MIN", "NH3_MIN", "NH3_N_MAX", "NH3_MAX", "NH3_N_AVG", "NH3_AVG"]:
    df_sub[col] = pd.to_numeric(df_sub[col], errors="coerce")
# convert NH3 columns to NH3_N columns if NH3_N columns are not available
df_sub['NH3_N_MIN'] = np.where(df_sub['NH3_N_MIN'].isna(), df_sub['NH3_MIN'], df_sub['NH3_N_MIN']) * (14.007/17.031)
df_sub['NH3_N_MAX'] = np.where(df_sub['NH3_N_MAX'].isna(), df_sub['NH3_MAX'], df_sub['NH3_N_MAX']) * (14.007/17.031)
df_sub['NH3_N_AVG'] = np.where(df_sub['NH3_N_AVG'].isna(), df_sub['NH3_AVG'], df_sub['NH3_N_AVG']) * (14.007/17.031)
df_sub['NH3_N_UNITS'] = np.where(df_sub['NH3_N_UNITS'].isna(), df_sub['NH3_UNITS'], df_sub['NH3_N_UNITS'])

# Calculate average when both min and max are available but avg is not
df_sub['NH3_N_AVG'] = np.where(df_sub['NH3_N_AVG'].isna() &
                                     ~df_sub['NH3_N_MIN'].isna() &  
                                        ~df_sub['NH3_N_MAX'].isna(),
                                        (df_sub['NH3_N_MIN'] + df_sub['NH3_N_MAX']) / 2,
                                        df_sub['NH3_N_AVG']) 
# drop NH3 columns
df_sub = df_sub.drop(columns=['NH3_MIN', 'NH3_MAX', 'NH3_AVG', 'NH3_UNITS'])
# Finalize and add to all_data
all_data['NH3_N'] = finalize_dataframe(df_sub, 'NH3_N')

# Check NO3_N data availability
df_sub = df_select[[
    'CWNS_ID',
    'NO3_N_MIN',
    'NO3_N_MAX',
    'NO3_N_AVG',
    'NO3_N_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no NO3_N data:")
print(df_sub[df_sub['NO3_N_MIN'].isna() &
                    df_sub['NO3_N_MAX'].isna() & 
                    df_sub['NO3_N_AVG'].isna()])
# remove rows with no NO3_N data
df_sub = df_sub[~(df_sub['NO3_N_MIN'].isna() & 
                               df_sub['NO3_N_MAX'].isna() & 
                                df_sub['NO3_N_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['NO3_N'] = finalize_dataframe(df_sub, 'NO3_N')

# Check NO2_N data availability
df_sub = df_select[[
    'CWNS_ID',
    'NO2_N_MIN',
    'NO2_N_MAX',
    'NO2_N_AVG',
    'NO2_N_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no NO2_N data:")
print(df_sub[df_sub['NO2_N_MIN'].isna() &
                    df_sub['NO2_N_MAX'].isna() & 
                    df_sub['NO2_N_AVG'].isna()])
# remove rows with no NO2_N data
df_sub = df_sub[~(df_sub['NO2_N_MIN'].isna() &
                                 df_sub['NO2_N_MAX'].isna() & 
                                  df_sub['NO2_N_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['NO2_N'] = finalize_dataframe(df_sub, 'NO2_N')

# Check NO3_N+NO2_N data availability
df_sub = df_select[[
    'CWNS_ID',
    'NO3_N+NO2_N_MIN',
    'NO3_N+NO2_N_MAX',
    'NO3_N+NO2_N_AVG',
    'NO3_N+NO2_N_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no NO3_N+NO2_N data:")
print(df_sub[df_sub['NO3_N+NO2_N_MIN'].isna() &
                    df_sub['NO3_N+NO2_N_MAX'].isna() & 
                    df_sub['NO3_N+NO2_N_AVG'].isna()])
# remove rows with no NO3_N+NO2_N data
df_sub = df_sub[~(df_sub['NO3_N+NO2_N_MIN'].isna() & 
                               df_sub['NO3_N+NO2_N_MAX'].isna() & 
                                df_sub['NO3_N+NO2_N_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['NO3_N+NO2_N'] = finalize_dataframe(df_sub, 'NO3_N+NO2_N')

# Check P_TOTAL data availability
df_sub = df_select[[
    'CWNS_ID',
    'P_TOTAL_MIN',
    'P_TOTAL_MAX',
    'P_TOTAL_AVG',
    'P_TOTAL_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no P_TOTAL data:")
print(df_sub[df_sub['P_TOTAL_MIN'].isna() &
                    df_sub['P_TOTAL_MAX'].isna() & 
                    df_sub['P_TOTAL_AVG'].isna()])
# remove rows with no P_TOTAL data
df_sub = df_sub[~(df_sub['P_TOTAL_MIN'].isna() & 
                               df_sub['P_TOTAL_MAX'].isna() &
                                df_sub['P_TOTAL_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['P_TOTAL'] = finalize_dataframe(df_sub, 'P_TOTAL')

# Check P_SOLID data availability
df_sub = df_select[[
    'CWNS_ID',
    'P_SOLID_MIN',
    'P_SOLID_MAX',
    'P_SOLID_AVG',
    'P_SOLID_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no P_SOLID data:")
print(df_sub[df_sub['P_SOLID_MIN'].isna() &
                    df_sub['P_SOLID_MAX'].isna() & 
                    df_sub['P_SOLID_AVG'].isna()])
# remove rows with no P_SOLID data
df_sub = df_sub[~(df_sub['P_SOLID_MIN'].isna() & 
                               df_sub['P_SOLID_MAX'].isna() &
                                df_sub['P_SOLID_AVG'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['P_SOLID'] = finalize_dataframe(df_sub, 'P_SOLID')

# Check ELECTRICITY_CONSUMED_ONSITE data availability
df_sub = df_select[[
    'CWNS_ID',
    'ELECTRICITY_CONSUMED_ONSITE__YEARLY_TOTAL',
    'ELECTRICITY_CONSUMED_ONSITE_YEARLY_TOTAL_UNITS',
]].copy() 
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no ELECTRICITY_CONSUMED_ONSITE data:")
print(df_sub[df_sub['ELECTRICITY_CONSUMED_ONSITE__YEARLY_TOTAL'].isna() &
             df_sub['ELECTRICITY_CONSUMED_ONSITE_YEARLY_TOTAL_UNITS'].isna()])

# remove rows with no ELECTRICITY_CONSUMED_ONSITE data
df_sub = df_sub[~(df_sub['ELECTRICITY_CONSUMED_ONSITE__YEARLY_TOTAL'].isna() &
                  df_sub['ELECTRICITY_CONSUMED_ONSITE_YEARLY_TOTAL_UNITS'].isna())].reset_index(drop=True)

# Convert ELECTRICITY_CONSUMED_ONSITE_YEARLY_TOTAL columns with values in MWh to kWh
df_sub['ELECTRICITY_CONSUMED_ONSITE__YEARLY_TOTAL'] = np.where(
    df_sub['ELECTRICITY_CONSUMED_ONSITE_YEARLY_TOTAL_UNITS'] == 'MWh',
    df_sub['ELECTRICITY_CONSUMED_ONSITE__YEARLY_TOTAL'] * 1000,
    df_sub['ELECTRICITY_CONSUMED_ONSITE__YEARLY_TOTAL']
)
df_sub['ELECTRICITY_CONSUMED_ONSITE_YEARLY_TOTAL_UNITS'] = np.where(
    df_sub['ELECTRICITY_CONSUMED_ONSITE_YEARLY_TOTAL_UNITS'] == 'MWh',
    'kWh', df_sub['ELECTRICITY_CONSUMED_ONSITE_YEARLY_TOTAL_UNITS']
)
# Merge flow data to calculate electricity consumed per cubic meter
df_sub = df_sub.merge(
    all_data['Flow'][['CWNS_ID', 'ACTUAL_FLOW_AVG', 'ACTUAL_FLOW_UNITS']],
    on='CWNS_ID', how='left'
)   
df_sub['Electricity_consumed_onsite_kWh_per_m3'] = np.where(
    ~df_sub['ELECTRICITY_CONSUMED_ONSITE__YEARLY_TOTAL'].isna() &
    ~df_sub['ACTUAL_FLOW_AVG'].isna() &
    (df_sub['ACTUAL_FLOW_UNITS'] == 'MGD'),
    df_sub['ELECTRICITY_CONSUMED_ONSITE__YEARLY_TOTAL'] / (df_sub['ACTUAL_FLOW_AVG'] * 365 * 3785.41),
    np.nan
)

# Finalize and add to all_data
all_data['Electricity_Consumed_Onsite'] = finalize_dataframe(df_sub, 'Electricity_Consumed_Onsite')

# Check ELECTRICITY_PURCHASED_FROM_UTILITY_YEARLY_TOTAL data availability
df_sub = df_select[[
    'CWNS_ID',
    'ELECTRICITY_PURCHASED_FROM_UTILITY_YEARLY_TOTAL',
    'ELECTRICITY_PURCHASED_FROM_UTILITY_YEARLY_TOTAL_UNITS',
    'Did_you_differentiate_between_the_total_energy_and_energy_purchased?']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no ELECTRICITY_PURCHASED_FROM_UTILITY data:")
print(df_sub[df_sub['ELECTRICITY_PURCHASED_FROM_UTILITY_YEARLY_TOTAL'].isna() &
             df_sub['ELECTRICITY_PURCHASED_FROM_UTILITY_YEARLY_TOTAL_UNITS'].isna()])

# remove rows with no ELECTRICITY_PURCHASED_FROM_UTILITY data
df_sub = df_sub[~(df_sub['ELECTRICITY_PURCHASED_FROM_UTILITY_YEARLY_TOTAL'].isna() &
                  df_sub['ELECTRICITY_PURCHASED_FROM_UTILITY_YEARLY_TOTAL_UNITS'].isna())].reset_index(drop=True)

# Merge flow data to calculate electricity purchased per cubic meter
df_sub = df_sub.merge(
    all_data['Flow'][['CWNS_ID', 'ACTUAL_FLOW_AVG', 'ACTUAL_FLOW_UNITS']],
    on='CWNS_ID', how='left'
)
df_sub['Electricity_purchased_from_utility_kWh_per_m3'] = np.where(
    ~df_sub['ELECTRICITY_PURCHASED_FROM_UTILITY_YEARLY_TOTAL'].isna() &
    ~df_sub['ACTUAL_FLOW_AVG'].isna() &
    (df_sub['ACTUAL_FLOW_UNITS'] == 'MGD'),
    df_sub['ELECTRICITY_PURCHASED_FROM_UTILITY_YEARLY_TOTAL'] / (df_sub['ACTUAL_FLOW_AVG'] * 365 * 3785.41),
    np.nan
)
# Finalize and add to all_data
all_data['Electricity_Purchased_From_Utility'] = finalize_dataframe(df_sub, 'Electricity_Purchased_From_Utility')

# Check ELECTRICITY_PRODUCED_ONSITE data availability
df_sub = df_select[[
    'CWNS_ID',
    'ELECTRICITY_PRODUCED_ONSITE_MIN',
    'ELECTRICITY_PRODUCED_ONSITE_MAX',
    'ELECTRICITY_PRODUCED_ONSITE_AVG',
    'ELECTRICITY_PRODUCED_ONSITE_NON_TOTAL_UNITS',
    'ELECTRICITY_PRODUCED_ONSITE__YEARLY_TOTAL',
    'ELECTRICITY_PRODUCED_ONSITE_YEARLY_TOTAL_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no ELECTRICITY_PRODUCED_ONSITE data:")
print(df_sub[df_sub['ELECTRICITY_PRODUCED_ONSITE_MIN'].isna() &
                    df_sub['ELECTRICITY_PRODUCED_ONSITE_MAX'].isna() & 
                    df_sub['ELECTRICITY_PRODUCED_ONSITE_AVG'].isna() &
                    df_sub['ELECTRICITY_PRODUCED_ONSITE__YEARLY_TOTAL'].isna()])
# remove rows with no ELECTRICITY_PRODUCED_ONSITE data
df_sub = df_sub[~(df_sub['ELECTRICITY_PRODUCED_ONSITE_MIN'].isna() & 
                               df_sub['ELECTRICITY_PRODUCED_ONSITE_MAX'].isna() & 
                                df_sub['ELECTRICITY_PRODUCED_ONSITE_AVG'].isna() &
                                df_sub['ELECTRICITY_PRODUCED_ONSITE__YEARLY_TOTAL'].isna())].reset_index(drop=True)
# Convert ELECTRICITY_PRODUCED_ONSITE_YEARLY_TOTAL columns with values in MWh to kWh
df_sub['ELECTRICITY_PRODUCED_ONSITE__YEARLY_TOTAL'] = np.where(
    df_sub['ELECTRICITY_PRODUCED_ONSITE_YEARLY_TOTAL_UNITS'] == 'MWh',
    df_sub['ELECTRICITY_PRODUCED_ONSITE__YEARLY_TOTAL'] * 1000,
    df_sub['ELECTRICITY_PRODUCED_ONSITE__YEARLY_TOTAL']
)  
df_sub['ELECTRICITY_PRODUCED_ONSITE_YEARLY_TOTAL_UNITS'] = np.where(
    df_sub['ELECTRICITY_PRODUCED_ONSITE_YEARLY_TOTAL_UNITS'] == 'MWh',
    'kWh', df_sub['ELECTRICITY_PRODUCED_ONSITE_YEARLY_TOTAL_UNITS']
) 
# Merge flow data to calculate electricity produced per m3
df_sub = df_sub.merge(
    all_data['Flow'][['CWNS_ID', 'ACTUAL_FLOW_AVG', 'ACTUAL_FLOW_UNITS']],
    on='CWNS_ID', how='left'
)  
df_sub['Electricity_produced_onsite_kWh_per_m3'] = np.where(
    ~df_sub['ELECTRICITY_PRODUCED_ONSITE__YEARLY_TOTAL'].isna() &
    ~df_sub['ACTUAL_FLOW_AVG'].isna() &
    (df_sub['ACTUAL_FLOW_UNITS'] == 'MGD'),
    df_sub['ELECTRICITY_PRODUCED_ONSITE__YEARLY_TOTAL'] / (df_sub['ACTUAL_FLOW_AVG'] * 365 * 3785.41),
    np.nan
)
# Finalize and add to all_data
all_data['Electricity_Produced_Onsite'] = finalize_dataframe(df_sub, 'Electricity_Produced_Onsite')

# Check NATURAL_GAS_PURCHASED_ONSITE data availability
df_sub = df_select[[
    'CWNS_ID',
    'NATURAL_GAS_PURCHASED_ONSITE_MIN',
    'NATURAL_GAS_PURCHASED_ONSITE_MAX',
    'NATURAL_GAS_PURCHASED_ONSITE_AVG',
    'NATURAL_GAS_PURCHASED_ONSITE_NON_TOTAL_UNITS',
    'NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL',
    'NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no NATURAL_GAS_PURCHASED_ONSITE data:")
print(df_sub[df_sub['NATURAL_GAS_PURCHASED_ONSITE_MIN'].isna() &
                    df_sub['NATURAL_GAS_PURCHASED_ONSITE_MAX'].isna() & 
                    df_sub['NATURAL_GAS_PURCHASED_ONSITE_AVG'].isna() &
                    df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'].isna()])
# remove rows with no NATURAL_GAS_PURCHASED_ONSITE data
df_sub = df_sub[~(df_sub['NATURAL_GAS_PURCHASED_ONSITE_MIN'].isna() & 
                               df_sub['NATURAL_GAS_PURCHASED_ONSITE_MAX'].isna() &
                                df_sub['NATURAL_GAS_PURCHASED_ONSITE_AVG'].isna() &
                                df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'].isna())].reset_index(drop=True)
# Convert NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL columns with values in cf, CCF, MCF, MMCF, kcf, scf, Therms to MMBtu
df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'] = np.where(
    df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS'].isin(['CCF', 'ccf']),
    df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'] * 0.1029,
    np.where(
        df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS'] == 'MCF',
        df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'] * 102.9,
        np.where(
            df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS'] == 'MMCF',
            df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'] * 102900,
            np.where(
                df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS'] == 'kcf',
                df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'] * 0.1029,
                np.where(
                    df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS'] == 'scf',
                    df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'] * 0.0001029,
                    np.where(
                        df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS'] == 'Therms',
                        df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'] * 0.1,
                        np.where(
                            df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS'] == 'cf',
                            df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'] * 0.0001029,
                            np.where(
                            df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS'] == 'CF',
                            df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'] * 0.0001029,
                        df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'],
                            )
                        )
                    )
                )
            )  
        )
    )
)
# convert GALLONS OF PROPANE (NOT NATURAL GAS) used on-site as a back-up fuel (to digester gas) for the raw sewage pump engines and for the boilers to MMBtu
###df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'] = np.where(
###    df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS'].isin(['GALLONS OF PROPANE (NOT NATURAL GAS) used on-site as a back-up fuel (to digester gas) for the raw sewage pump engines and for the boilers']),
###    df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'] * 0.0915,
###    df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL']
###)

df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS'] = np.where(
    df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS'].isin(['CCF', 'ccf', 'kcf', 'scf', 'Therms', 'MCF', 'MMCF', 'GALLONS OF PROPANE (NOT NATURAL GAS) used on-site as a back-up fuel (to digester gas) for the raw sewage pump engines and for the boilers']),
    'MMBtu', df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL_UNITS']
)
# Merge flow data to calculate natural gas consumed per cubic meter
df_sub = df_sub.merge(
    all_data['Flow'][['CWNS_ID', 'ACTUAL_FLOW_AVG', 'ACTUAL_FLOW_UNITS']],
    on='CWNS_ID', how='left'
)
df_sub['NATURAL_GAS_PURCHASED_onsite_MMBtu_per_m3'] = np.where(
    ~df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'].isna() &
    ~df_sub['ACTUAL_FLOW_AVG'].isna() &
    (df_sub['ACTUAL_FLOW_UNITS'] == 'MGD'),
    df_sub['NATURAL_GAS_PURCHASED_ONSITE_YEARLY_TOTAL'] / (df_sub['ACTUAL_FLOW_AVG'] * 365 * 3785.41),
    np.nan
)
# convert MMBTU to MJ
df_sub['NATURAL_GAS_PURCHASED_onsite_MJ_per_m3'] = df_sub['NATURAL_GAS_PURCHASED_onsite_MMBtu_per_m3'] * 1055.06

# Finalize and add to all_data
all_data['NATURAL_GAS_PURCHASED_ONSITE'] = finalize_dataframe(df_sub, 'NATURAL_GAS_PURCHASED_ONSITE')

# Check ANAEROBIC_DIGESTER_CONDITION data availability
df_sub = df_select[[
    'CWNS_ID',
    'ANAEROBIC_DIGESTER_CONDITION',
    'QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL',
    'QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL_UNITS',
    'BIOGAS_UTILIZATION',
    'BIOGAS_UTILIZATION_LOCATION']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no ANAEROBIC_DIGESTER_CONDITION data:")
print(df_sub[df_sub['ANAEROBIC_DIGESTER_CONDITION'].isna() &
                    df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL'].isna() & 
                    df_sub['BIOGAS_UTILIZATION'].isna() &
                    df_sub['BIOGAS_UTILIZATION_LOCATION'].isna()])
# remove rows with no ANAEROBIC_DIGESTER_CONDITION data or not measured
df_sub = df_sub[~(df_sub['ANAEROBIC_DIGESTER_CONDITION'].isna() & 
                               df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL'].isna() &
                                df_sub['BIOGAS_UTILIZATION'].isna() &
                                df_sub['BIOGAS_UTILIZATION_LOCATION'].isna())].reset_index(drop=True)
df_sub = df_sub[~df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL'].isin(['NOT MEASURED'])]
# Convert cf, CF, scf, scfm, kcf, KSCF to MMBtu
hhv_methane = 1068  # Btu/scf # R&D GREET 2024
methane_fraction_in_biogas = 0.6  # 60%
biogas_conv_factor = hhv_methane * methane_fraction_in_biogas / 1e6  # MMBtu/scf
df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL'] = np.where(
    df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL_UNITS'].isin(['cf', 'CF','scf', 'scfm', 'SCF']),
    df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL'] * biogas_conv_factor,
    np.where(
            df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL_UNITS'].isin(['kcf', 'KSCF']),
            df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL'] * biogas_conv_factor * 1000,
            df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL']
    )
)
df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL_UNITS'] = np.where(
    df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL_UNITS'].isin(['cf', 'CF','scf', 'scfm', 'SCF', 'kcf', 'KSCF']),
    'MMBtu', df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL_UNITS']
)
# Merge flow data to calculate biogas generated per m3
df_sub = df_sub.merge(
    all_data['Flow'][['CWNS_ID', 'ACTUAL_FLOW_AVG', 'ACTUAL_FLOW_UNITS']],
    on='CWNS_ID', how='left'
)
df_sub['Biogas_generated_onsite_MMBtu_per_m3'] = np.where(
    ~df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL'].isna() &
    ~df_sub['ACTUAL_FLOW_AVG'].isna() &
    (df_sub['ACTUAL_FLOW_UNITS'] == 'MGD'),
    df_sub['QUANTITY_BIOGAS_GENERATED_YEARLY_TOTAL'] / (df_sub['ACTUAL_FLOW_AVG'] * 365 * 3785.41),
    np.nan
)
# convert MMBTU to MJ
df_sub['Biogas_generated_onsite_MJ_per_m3'] = df_sub['Biogas_generated_onsite_MMBtu_per_m3'] * 1055.06

# Finalize and add to all_data
all_data['ANAEROBIC_DIGESTER_BIOGASS'] = finalize_dataframe(df_sub, 'ANAEROBIC_DIGESTER_BIOGASS')

# Check CHEMICALS_PRODUCED_ONSITE data availability
df_sub = df_select[[
    'CWNS_ID',
    'CHEMICALS_PRODUCED_ONSITE']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)
print("Rows with no CHEMICALS_PRODUCED_ONSITE data:")
print(df_sub[df_sub['CHEMICALS_PRODUCED_ONSITE'].isna()])
# remove rows with no CHEMICALS_PRODUCED_ONSITE data
df_sub = df_sub[~(df_sub['CHEMICALS_PRODUCED_ONSITE'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['CHEMICALS_PRODUCED_ONSITE'] = finalize_dataframe(df_sub, 'CHEMICALS_PRODUCED_ONSITE')

# Check SLUDGE_HANDLING_METHOD data availability
df_sub = df_select[[
    'CWNS_ID',
    'SLUDGE_HANDLING_METHOD']].copy()
df_sub['CWNS_ID'] = df_sub['CWNS_ID'].astype(int)   
print("Rows with no SLUDGE_HANDLING_METHOD data:")
print(df_sub[df_sub['SLUDGE_HANDLING_METHOD'].isna()])
# remove rows with no SLUDGE_HANDLING_METHOD data
df_sub = df_sub[~(df_sub['SLUDGE_HANDLING_METHOD'].isna())].reset_index(drop=True)
# Finalize and add to all_data
all_data['SLUDGE_HANDLING_METHOD'] = finalize_dataframe(df_sub, 'SLUDGE_HANDLING_METHOD')

# Print summary of all DataFrames in all_data
print(f"\n=== SUMMARY OF ALL DATAFRAMES ===")
for df_name, df in all_data.items():
    print(f"{df_name}: {len(df)} rows, {len(df.columns)} columns, {df['CWNS_ID'].nunique()} unique CWNS_IDs")

# Process all tabs using batch function
def process_all_tabs_batch(all_data_dict, path_out=path_out, fi_out=fi_out):
    """
    Process all tabs in a single batch operation to avoid file conflicts
    
    Parameters:
    - all_data_dict: Dictionary containing {sheet_name: dataframe} pairs
    - path_out: Output directory path
    - fi_out: Output filename
    """
    
    file_path = os.path.join(path_out, fi_out)
    
    # Write all tabs at once
    with pd.ExcelWriter(file_path, engine='openpyxl', mode='w') as writer:
        for sheet_name, dataframe in all_data_dict.items():
            # Ensure sheet name is valid (Excel has 31 character limit)
            valid_sheet_name = sheet_name[:31] if len(sheet_name) > 31 else sheet_name
            dataframe.to_excel(writer, sheet_name=valid_sheet_name, index=False)
            print(f"Tab '{valid_sheet_name}' written successfully")
    
    print(f"All {len(all_data_dict)} tabs written to {fi_out}")

# Call the batch function to save all data
if write_to_excel_tab:
    process_all_tabs_batch(all_data)

       CWNS_ID PREDICTED_WRRF_TT_CODE  TT_IDENTIFIED Assigned TT
0  17000721001                   *AG2              2    *G1, *A2
1  26000596001                *B6, *B              1         *B6
2  11000001001                 *AEF2e              2    F1e, *A2
3  25000128001                   *B1e              2   *B1e, *A2
4   6004010001                   *B1e              2   *B1e, *A2
Finalizing 'TTs' DataFrame...
  Initial: 36 rows, 4 columns
  Final: 36 rows, 4 columns
  ✓ All CWNS_IDs are unique
Rows with no flow data:
Empty DataFrame
Columns: [CWNS_ID, FACILITY_ID, FACILITY_NAME, LATITUDE, LONGITUDE, CITY, STATE_CODE, AUTHORITY_NAME, COUNTY_NAME, DESIGN_FLOW, DESIGN_FLOW_UNITS, ACTUAL_FLOW_MIN, ACTUAL_FLOW_MAX, ACTUAL_FLOW_AVG, ACTUAL_FLOW_UNITS]
Index: []
Finalizing 'Flow' DataFrame...
  Initial: 37 rows, 15 columns
  Final: 37 rows, 15 columns
  ✓ All CWNS_IDs are unique
Rows with no BOD5 data:
        CWNS_ID  BOD5_MIN  BOD5_MAX  BOD5_AVG BOD5_UNITS
1   26000596001       NaN  

In [ ]:
# Create merged dataframes - Simple left join approach
print("Starting simple merge process...")

# Start with Flow data as the base
df_merged = all_data['Flow'].copy()
print(f"Base DataFrame 'Flow' has {len(df_merged)} rows and {len(df_merged.columns)} columns")

# Get all other DataFrames to merge (excluding Flow)
dataframes_to_merge = [key for key in all_data.keys() if key != 'Flow']

# Simple left join with each DataFrame
for df_name in dataframes_to_merge:
    print(f"Merging '{df_name}'...")
    
    df_to_merge = all_data[df_name].copy()
    
    # Drop flow columns if they exist (to avoid duplicates)
    df_to_merge = df_to_merge.drop(columns=['ACTUAL_FLOW_AVG', 'ACTUAL_FLOW_UNITS'], errors='ignore')
    
    # Simple left join
    df_merged = df_merged.merge(df_to_merge, on='CWNS_ID', how='left')
    
    print(f"  After merging '{df_name}': {len(df_merged)} rows and {len(df_merged.columns)} columns")

# Electricity purchased kwh to MJ
df_merged['Electricity_purchased_from_utility_MJ_per_m3'] =\
df_merged['Electricity_purchased_from_utility_kWh_per_m3'] * 3.6

# Electricity consumed kWh to MJ
df_merged['Electricity_consumed_onsite_MJ_per_m3'] =\
df_merged['Electricity_consumed_onsite_kWh_per_m3'] * 3.6

# Electricity produced kWh to MJ
df_merged['Electricity_produced_onsite_MJ_per_m3'] =\
df_merged['Electricity_produced_onsite_kWh_per_m3'] * 3.6

# Calculate Total energy purchased_MJ_per_m3_survey
df_merged['Total energy purchased_MJ_per_m3_survey'] = df_merged['NATURAL_GAS_PURCHASED_onsite_MJ_per_m3'] +\
    df_merged['Electricity_purchased_from_utility_MJ_per_m3']

print(f"\nFinal merged DataFrame:")
print(f"  Total rows: {len(df_merged)}")
print(f"  Total columns: {len(df_merged.columns)}")
print(f"  Unique CWNS_IDs: {df_merged['CWNS_ID'].nunique()}")

# Add to all_data
all_data['Merged_All_Data'] = df_merged.copy()

df_merged.to_excel(os.path.join(path_out, 'survey_data_one.xlsx'), index=False)


Starting simple merge process...
Base DataFrame 'Flow' has 37 rows and 15 columns
Merging 'TTs'...
  After merging 'TTs': 37 rows and 18 columns
Merging 'BOD5'...
  After merging 'BOD5': 37 rows and 22 columns
Merging 'CBOD5'...
  After merging 'CBOD5': 37 rows and 26 columns
Merging 'SS'...
  After merging 'SS': 37 rows and 30 columns
Merging 'VSS'...
  After merging 'VSS': 37 rows and 34 columns
Merging 'TS'...
  After merging 'TS': 37 rows and 38 columns
Merging 'TSS'...
  After merging 'TSS': 37 rows and 42 columns
Merging 'VTS'...
  After merging 'VTS': 37 rows and 46 columns
Merging 'TKN'...
  After merging 'TKN': 37 rows and 50 columns
Merging 'TN'...
  After merging 'TN': 37 rows and 54 columns
Merging 'NH3_N'...
  After merging 'NH3_N': 37 rows and 58 columns
Merging 'NO3_N'...
  After merging 'NO3_N': 37 rows and 62 columns
Merging 'NO2_N'...
  After merging 'NO2_N': 37 rows and 66 columns
Merging 'NO3_N+NO2_N'...
  After merging 'NO3_N+NO2_N': 37 rows and 70 columns
Merging 

In [6]:
# Merge TT data from Abbadi et al. 2024 paper
df_merged = df_merged.merge(df_SI_C[['CWNS code', 'treatment train']], left_on='CWNS_ID', right_on='CWNS code', how='left').rename(columns={'treatment train': 'Treatment_Train_byLiterature'}).copy()

df_merged['Treatment_Train_byLiterature'] = (df_merged['Treatment_Train_byLiterature'].astype(str)
             .str.strip()
             .str.strip('[]')                      # remove [ ]
             .str.replace("'", "", regex=False)    # remove single quotes
             .str.replace('"', "", regex=False)    # optional: remove double quotes
             .str.replace(r"\s*,\s*", ", ", regex=True)  # normalize comma spacing
             .str.strip())

# Save the updated merged dataframe with treatment train info
df_merged.to_excel(os.path.join(path_out, 'survey_data_one.xlsx'), index=False)

df_merged.drop(columns=['CWNS code'], inplace=True)

In [7]:
# Load water quality standard data

criteria_data_path = 'C:/Users/skar/Box/_Saura_Self/Proj - Water tool analysis/data'
criteria_data_file = 'EPA state-specific water quality standards under CWA criteria-search-tool-data_vJuly 11 2025.xlsx'
df_criteria = pd.read_excel(os.path.join(criteria_data_path, criteria_data_file), sheet_name='Criteria Data', 
                           #dtype={'CRITERION_VALUE': float, 'POLLUTANT_NAME': str, 'STD_POLLUTANT_NAME': str, 'ENTITY_NAME': str, 'UNIT_NAME': str}
                           )

# Dict to map pollutant column names to criteria pollutant names
pollutants_dict = {
'BOD5_AVG' : ['biochemical oxygen demand (bod)', 'biochemical oxygen demand',],
'CBOD5_AVG' : ['chemical oxygen demand (COD)',],
'SS_AVG' : ['suspended solids', 'suspended sediment', 'solids suspended and turbidity'],
'VSS_AVG' : [],
'TS_AVG' : [],
'TSS_AVG' : ['total suspended solids'],
'VTS_AVG' : [],
'TKN_AVG' : ['total kjehldal nitrogen'], # total kjehldal nitrogen
'TN_AVG' : ['total nitrogen', 'nitrogen', 'total dissolved nitrogen'],
'NH3_N_AVG' : ['ammonia', 'ammonia, un-ionized', 'ammonia and ammonium'], # 'total ammonia nitrogen' has values higher than influent conc. at some cases, so ignoring it unless further clarity is received
'NO3_N_AVG' : ['nitrate (expressed as n)', 'nitrates', 'nitrate'],
'NO2_N_AVG' : ['nitrites', 'nitrite', 'nitrite (expressed as n)'],
'NO3_N+NO2_N_AVG' : ['nitrate + nitrite', 'nitrate and nitrite (expressed as n)', 'nitrate and nitrite', 'nitrates+nitrites', 'total nitrate and nitrite'],
'P_TOTAL_AVG' : ['total phosphorus', 'phosphorous (yellow)', 'phosphorus, elemental', 'total dissolved phosphorus'],
'P_SOLID_AVG' : [],
}

# Keep rows with pollutants of interest
df_criteria_sub = df_criteria[df_criteria['POLLUTANT_NAME'].str.lower().isin([item for sublist in pollutants_dict.values() for item in sublist])].copy()

# Down-select use classes for surface water
use_classes_to_keep = pd.read_excel(os.path.join(criteria_data_path, 'EPA criteria use case selections.xlsx'))
use_classes_to_keep = use_classes_to_keep.loc[use_classes_to_keep['To_include'] == 'Y']
df_criteria_sub = df_criteria_sub[df_criteria_sub['USE_CLASS_NAME_LOCATION_ETC'].isin(use_classes_to_keep['USE_CLASS_NAME_LOCATION_ETC'])].reset_index(drop=True)

# Identify pattern of criterion values such as 93,330 and convert to numeric
df_criteria_sub['CRITERION_VALUE'] = df_criteria_sub['CRITERION_VALUE'].astype(str).str.replace(',', '')

# If criterion value is a range, e.g. 40 - 105, separate the column into min and max
df_criteria_sub[['CRITERION_VALUE_MIN', 'CRITERION_VALUE_MAX']] = (
    df_criteria_sub['CRITERION_VALUE']
    .str.split('-', expand=True)
    .reindex(columns=[0, 1])  # Ensure two columns are created
)

# Convert the split columns to numeric, coercing errors to NaN
df_criteria_sub['CRITERION_VALUE_MIN'] = pd.to_numeric(df_criteria_sub['CRITERION_VALUE_MIN'], errors='coerce')
df_criteria_sub['CRITERION_VALUE_MAX'] = pd.to_numeric(df_criteria_sub['CRITERION_VALUE_MAX'], errors='coerce')

# drop rows where both min and max are NaN
df_criteria_sub = df_criteria_sub[~(df_criteria_sub['CRITERION_VALUE_MIN'].isna() & df_criteria_sub['CRITERION_VALUE_MAX'].isna())].reset_index(drop=True)

# Convert CRITERION_VALUE_MIN, CRITERION_VALUE_MAX units µg/l, tons/million cubic meters of water, parts per billion (ppb) units to mg/L
unit_conversion = {
    'µg/l': 1e-3,
    'tons/million cubic meters of water': 0.90718474,  # 1 ton/million cubic meters = 0.90718474 mg/L
    'parts per billion (ppb)': 1e-3,
    'mg/L': 1,
    'mg/l': 1,
    'mg/l as n': 1,
    'ug/L': 1e-3,
    'ppb': 1e-3,    
}

# Standardize the target unit we want after conversion
target_unit = "mg/L"

# Map conversion factor (default 1 if not found)
df_criteria_sub['conversion_factor'] = (
    df_criteria_sub['UNIT_NAME'].map(unit_conversion).fillna(1)
)

# Apply conversion
df_criteria_sub['CRITERION_VALUE_MIN'] = (
    df_criteria_sub['CRITERION_VALUE_MIN'] * df_criteria_sub['conversion_factor']
)
df_criteria_sub['CRITERION_VALUE_MAX'] = (
    df_criteria_sub['CRITERION_VALUE_MAX'] * df_criteria_sub['conversion_factor']
)

# Update units only if a conversion was applied (factor != 1)
df_criteria_sub['UNIT_min_max_CONVERTED'] = np.where(
    df_criteria_sub['conversion_factor'] != 1,
    target_unit,  # converted unit
    df_criteria_sub['UNIT_NAME']  # keep original
)

# Drop helper column
df_criteria_sub = df_criteria_sub.drop(columns=['conversion_factor'])

# replace mg/l with mg/L
df_criteria_sub['UNIT_min_max_CONVERTED'] = df_criteria_sub['UNIT_min_max_CONVERTED'].replace({'mg/l': 'mg/L'})

# Aggregate criteria by ENTITY_NAME, ENTITY_ABBR, POLLUTANT_NAME, UNIT_NAME, USE_CLASS_NAME_LOCATION_ETC.
# For the min, take the minimum of CRITERION_VALUE_MIN, for the max, take the maximum of CRITERION_VALUE_MAX
df_criteria_sub_loc = df_criteria_sub.groupby(
    ['ENTITY_NAME', 'ENTITY_ABBR', 'POLLUTANT_NAME', 'UNIT_min_max_CONVERTED']
).agg({
    'CRITERION_VALUE_MIN': 'min',
    'CRITERION_VALUE_MAX': 'max'
}).reset_index()

# Calculate average column for those that have both min and max or assign min or max if only one is present
def calculate_average(row):
    if not pd.isna(row['CRITERION_VALUE_MIN']) and not pd.isna(row['CRITERION_VALUE_MAX']):
        return (row['CRITERION_VALUE_MIN'] + row['CRITERION_VALUE_MAX']) / 2
    elif not pd.isna(row['CRITERION_VALUE_MIN']):
        return row['CRITERION_VALUE_MIN']
    elif not pd.isna(row['CRITERION_VALUE_MAX']):
        return row['CRITERION_VALUE_MAX']
    else:
        return np.nan
df_criteria_sub_loc['CRITERION_VALUE_AVG'] = df_criteria_sub_loc.apply(calculate_average, axis=1)

df_criteria_sub.to_excel(os.path.join(path_out, 'Criteria_select_params.xlsx'), index=False)
df_criteria_sub_loc.to_excel(os.path.join(path_out, 'Criteria_select_params_summary_by_loc.xlsx'), index=False)


In [8]:
# Merge criteria data with survey data
df_criteria_sub_loc_1 = df_criteria_sub_loc.copy()
# Keep only the specified columns
df_criteria_sub_loc_1 = df_criteria_sub_loc[["ENTITY_NAME", "ENTITY_ABBR", "POLLUTANT_NAME", "UNIT_min_max_CONVERTED", "CRITERION_VALUE_AVG"]]
df_criteria_sub_loc_1.rename(columns={"UNIT_min_max_CONVERTED": "CRITERION_VALUE_UNITS"}, inplace=True)

# Spread POLLUTANT_NAME into columns and use CRITERION_VALUE_AVG as values
df_criteria_sub_loc_1 = df_criteria_sub_loc_1.pivot(index=["ENTITY_NAME", "ENTITY_ABBR", "CRITERION_VALUE_UNITS"], columns="POLLUTANT_NAME", values="CRITERION_VALUE_AVG").reset_index()

# Add prefix to all columns to identify NPDES criteria
df_criteria_sub_loc_1 = df_criteria_sub_loc_1.add_prefix('NPDES_')

# Collapse 'ammonia', 'ammonia, un-ionized' criteria to ammonia
df_criteria_sub_loc_1['NPDES_ammonia'] = df_criteria_sub_loc_1[['NPDES_ammonia', 'NPDES_ammonia, un-ionized']].min(axis=1)

# Collapse 'nitrate (expressed as n)', 'nitrates', 'nitrate' criteria to nitrate
df_criteria_sub_loc_1['NPDES_nitrate'] = df_criteria_sub_loc_1[['NPDES_nitrate', 'NPDES_nitrate (expressed as n)', 'NPDES_nitrates',]].min(axis=1)   

# Collapse 'nitrites', 'nitrite', 'nitrite (expressed as n)' criteria to nitrite
df_criteria_sub_loc_1['NPDES_nitrite'] = df_criteria_sub_loc_1[['NPDES_nitrite', 'NPDES_nitrites']].min(axis=1)

# drop the original columns that were collapsed
df_criteria_sub_loc_1.drop(columns=['NPDES_ammonia, un-ionized',
                                    'NPDES_nitrate (expressed as n)', 'NPDES_nitrates',
                                    'NPDES_nitrites'
                                    ], inplace=True)

# Merged survey data
df_merged_criteria = df_merged.merge(df_criteria_sub_loc_1, left_on='STATE_CODE', right_on='NPDES_ENTITY_ABBR', how='left').reset_index(drop=True)

df_merged_criteria.to_excel(os.path.join(path_out, 'survey_data_with_criteria.xlsx'), index=False)

In [9]:
# Process NPDES data as received from Abby
df_npdes = pd.read_excel(os.path.join(path_data_2, 'npdes_data_for_saura (1).xlsx'), sheet_name='npdes_data_for_saura (1)')
df_map = pd.read_excel(os.path.join(path_data_2, 'REF_Parameter_Abby_SK.xlsx'), sheet_name='REF_Parameter_Abby_SK')

df_npdes = df_npdes.merge(df_map[['PARAMETER_DESC', 'Survey_parameter_mapping']], on='PARAMETER_DESC', how='left')

# convert Nitrogen, ammonia, total [as NH3] to as NH3-N by multiplying by 0.822
mask = df_npdes['PARAMETER_DESC'] == 'Nitrogen, ammonia, total [as NH3]'
df_npdes.loc[mask, 'LIMIT_VALUE_STANDARD_UNITS'] *= 0.822

# convert Nitrogen, ammonia total [as NH4] to as NH4-N by multiplying by 0.78
mask = df_npdes['PARAMETER_DESC'] == 'Nitrogen, ammonia total [as NH4]'
df_npdes.loc[mask, 'LIMIT_VALUE_STANDARD_UNITS'] *= 0.78

# remove rows without Survey_parameter_mapping
df_npdes = df_npdes[~df_npdes['Survey_parameter_mapping'].isna()].reset_index(drop=True)

# remove rows with STATISTICAL_BASE_LONG_DESC having 'Maximum'
df_npdes = df_npdes[~df_npdes['STATISTICAL_BASE_LONG_DESC'].str.contains('Maximum', case=False, na=False)].reset_index(drop=True)

# calculate average
df_npdes = df_npdes.groupby(['CWNS_ID', 'Survey_parameter_mapping', 'STANDARD_UNIT_DESC', 'LIMIT_VALUE_QUALIFIER_CODE'])['LIMIT_VALUE_STANDARD_UNITS'].mean().reset_index()

print(df_npdes.shape)

# In case there are multiple 'standard_unit_desc' entries by different units for the same parameter grouped by UIDs ['CWNS_ID', 'value_received_year', 'Survey_parameter_mapping'], keep one with mg/L units if available, or by kg/d if mg/L is not available, or by lb/d if kg/d is not available, or keep the first one if none of the preferred units are available
# Implementing the logic to prioritize 'standard_unit_desc' entries
def prioritize_units(group):
    # Define the preferred order of units
    preferred_units = ['mg/L', '%', 'kg/d', 'lb/d']
    
    # Filter the group based on the preferred units
    for unit in preferred_units:
        if unit in group['STANDARD_UNIT_DESC'].values:
            return group[group['STANDARD_UNIT_DESC'] == unit].iloc[0]
    
    # If none of the preferred units are available, keep the first entry
    return group.iloc[0]

# Apply the prioritization logic to the grouped DataFrame
df_npdes = df_npdes.groupby(['CWNS_ID', 'Survey_parameter_mapping'], as_index=False).apply(prioritize_units)

# Reset the index to ensure the result is a proper DataFrame
df_npdes = df_npdes.reset_index(drop=True)
print(df_npdes.shape)

df_npdes.to_excel(os.path.join(path_out, 'NPDES_limits_Abby_processed.xlsx'), index=False)

(225, 5)
(103, 5)


In [10]:
# Merge df_npdes with df_merged_criteria to compare NPDES limits with criteria and survey values

# Concat unit to Survey_parameter_mapping as suffix and spread Survey_parameter_mapping into columns and use LIMIT_VALUE_STANDARD_UNITS as values while adding prefix to the columns "NPDES2_"
df_npdes['Survey_parameter_mapping'] = df_npdes['Survey_parameter_mapping'] + '_' + df_npdes['STANDARD_UNIT_DESC']
df_npdes_wide = df_npdes.pivot(index='CWNS_ID', columns='Survey_parameter_mapping', values='LIMIT_VALUE_STANDARD_UNITS').reset_index()
df_npdes_wide = df_npdes_wide.rename(columns={col: f"NPDES2_{col}" for col in df_npdes_wide.columns if col not in ['CWNS_ID']})

# merge based on CWNS_IDs
df_merged_criteria_npdes = df_merged_criteria.merge(df_npdes_wide, on='CWNS_ID', how='left').reset_index(drop=True)

# For columns with units of _% calculate the percentage based on the influent parameter value if available for the parameter

# identify columns with _% units
percent_columns = [col for col in df_merged_criteria_npdes.columns if col.startswith('NPDES2_') and col.endswith('_%')]
if percent_columns:
    for col in percent_columns:
        # extract the parameter name by removing the prefix and suffix
        param_name = col.replace('NPDES2_', '').replace('_%', '')
        #influent_col = param_name + '_AVG'
        
        if param_name in df_merged_criteria_npdes.columns:
            # mask to identify rows where both the percentage column and the influent parameter column are not null
            mask = ~df_merged_criteria_npdes[col].isna() & ~df_merged_criteria_npdes[param_name].isna()
            # for the masked column, calculate physical value based on percentage value and influent parameter value
            df_merged_criteria_npdes.loc[mask, f"{col}_calculated"] = (1 - (df_merged_criteria_npdes.loc[mask, col] / 100) ) * df_merged_criteria_npdes.loc[mask, param_name]
            # get the unit of the mask rows of param_name
            unit_col_name = param_name + '_UNITS'
            # for BOD5_AVG, the unit column is BOD5_UNITS instead of BOD5_AVG_UNITS, so check if param_name+'_UNITS' exists, if not check if param_name.replace('_AVG', '')+'_UNITS' exists and use that as unit_col_name
            if unit_col_name not in df_merged_criteria_npdes.columns:
                unit_col_name = param_name.replace('_AVG', '') + '_UNITS'
            influent_unit = df_merged_criteria_npdes.loc[mask, unit_col_name]
            # check if unique of influent_unit is mg/L, if so rename the col to have mg/L as suffix, if not print the units and post error message
            if influent_unit.nunique() == 1 and influent_unit.iloc[0] == 'mg/L':
                df_merged_criteria_npdes.rename(columns={f"{col}_calculated": f"{col}_calculated_mg_per_L"}, inplace=True)
            else:
                print(f"Unique units for influent parameter '{param_name}' are not consistent or not mg/L: {influent_unit.unique()}")

            # drop the original percentage column after calculation
            drop_percentage_col = True
            if drop_percentage_col:
                df_merged_criteria_npdes.drop(columns=[col], inplace=True)
        else:
            print(f"Influent column '{param_name}' not found for percentage calculation of '{col}'")
else:
    print("No percentage columns found for calculation")



In [11]:
# Merge NPDES and NPDES2 columns per mapping dictionary parameter by keeping the more stringent value (lower value for pollutants) and create new columns with suffix _NPDES_final and add a helper column which value is kept or if both were available and one is kept.

map_npdes_1_2 = {
'NPDES_ammonia' : 'NPDES2_NH3_N_AVG_mg/L',
#'NPDES_nitrate'
'NPDES_nitrates+nitrites'
'NPDES_nitrite' : 'NPDES2_NO2_N_AVG_mg/L',
'NPDES_total nitrogen' : 'NPDES2_TN_AVG_mg/L',
'NPDES_total phosphorus' : 'NPDES2_P_TOTAL_AVG_mg/L',
#'NPDES2_BOD5_AVG_%'
#'NPDES2_BOD5_AVG_mg/L'
#'NPDES2_CBOD5_AVG_mg/L'
#'NPDES2_TSS_AVG_mg/L'
#'NPDES2_BOD5_AVG_%_calculated_mg_per_L'
}

# merge based on the mapping dictionary
for npdes_col, npdes2_col in map_npdes_1_2.items():
    if npdes_col in df_merged_criteria_npdes.columns and npdes2_col in df_merged_criteria_npdes.columns:
        # create new column with the more stringent value (lower value for pollutants)
        df_merged_criteria_npdes[f"{npdes_col}_NPDES_final"] = df_merged_criteria_npdes[[npdes_col, npdes2_col]].min(axis=1)
        # create helper column to indicate which value is kept and why
        def indicate_kept_value(row):
            if pd.isna(row[npdes_col]) and not pd.isna(row[npdes2_col]):
                return f"kept {npdes2_col} as {npdes_col} is missing"
            elif not pd.isna(row[npdes_col]) and pd.isna(row[npdes2_col]):
                return f"kept {npdes_col} as {npdes2_col} is missing"
            elif not pd.isna(row[npdes_col]) and not pd.isna(row[npdes2_col]):
                if row[npdes_col] < row[npdes2_col]:
                    return f"kept {npdes_col} as more stringent with npdes value {row[npdes_col]} vs npdes2 value {row[npdes2_col]}"
                elif row[npdes2_col] < row[npdes_col]:
                    return f"kept {npdes2_col} as more stringent with npdes2 value {row[npdes2_col]} vs npdes value {row[npdes_col]}"
                else:
                    return "both values equal"
            else:
                return "both values missing"
        df_merged_criteria_npdes[f"{npdes_col}_NPDES_final_helper"] = df_merged_criteria_npdes.apply(indicate_kept_value, axis=1)
    else:
        print(f"Columns '{npdes_col}' or '{npdes2_col}' not found for merging")

# save the final merged dataframe with NPDES criteria
df_merged_criteria_npdes.to_excel(os.path.join(path_out, 'survey_NPDES2.xlsx'), index=False)
# print all column names
#for name in df_merged_criteria_npdes.columns:
#    print(name)

Columns 'NPDES_nitrates+nitritesNPDES_nitrite' or 'NPDES2_NO2_N_AVG_mg/L' not found for merging


In [12]:
# Process DMR data for CWNS_IDs x years x parameters

# DMR reported data set
df2_all_years = pd.read_excel(os.path.join(path_DMR, 'DMR_effluent_data.xlsx'), sheet_name='DMR_effluent_data')

df_dmr = df2_all_years[['CWNS_ID', 'value_received_year', 'Survey_parameter_mapping', 'standard_unit_desc', 'dmr_value_standard_units']]

df_dmr.shape

df_dmr = df_dmr.groupby(['CWNS_ID', 'value_received_year', 'Survey_parameter_mapping', 'standard_unit_desc'])['dmr_value_standard_units'].mean().reset_index()
print(df_dmr.shape)

# In case there are multiple 'standard_unit_desc' entries by different units for the same parameter grouped by UIDs ['CWNS_ID', 'value_received_year', 'Survey_parameter_mapping'], keep one with mg/L units if available, or by kg/d if mg/L is not available, or by lb/d if kg/d is not available, or keep the first one if none of the preferred units are available
# Implementing the logic to prioritize 'standard_unit_desc' entries
def prioritize_units(group):
    # Define the preferred order of units
    preferred_units = ['mg/L', 'kg/d', 'lb/d']
    
    # Filter the group based on the preferred units
    for unit in preferred_units:
        if unit in group['standard_unit_desc'].values:
            return group[group['standard_unit_desc'] == unit].iloc[0]
    
    # If none of the preferred units are available, keep the first entry
    return group.iloc[0]

# Apply the prioritization logic to the grouped DataFrame
df_dmr = df_dmr.groupby(['CWNS_ID', 'value_received_year', 'Survey_parameter_mapping'], as_index=False).apply(prioritize_units)

# Reset the index to ensure the result is a proper DataFrame
df_dmr = df_dmr.reset_index(drop=True)
print(df_dmr.shape)

df_dmr = df_dmr.pivot(index=['CWNS_ID', 'value_received_year', 'standard_unit_desc'], 
                   columns='Survey_parameter_mapping', 
                   values='dmr_value_standard_units').reset_index()

# add prefix DMR_ to all columns from df_dmr except CWNS_ID, value_received_year, standard_unit_desc
df_dmr = df_dmr.rename(columns={col: f"DMR_{col}" for col in df_dmr.columns if col not in ['CWNS_ID', 'value_received_year', 'standard_unit_desc']})
# remove rows without values in any of the DMR columns
dmr_columns = [col for col in df_dmr.columns if col.startswith('DMR_')]
df_dmr = df_dmr.dropna(subset=dmr_columns, how='all').reset_index(drop=True)

df_dmr.to_excel(os.path.join(path_out, 'DMR_reported_data.xlsx'), index=False)

# DMR limits data set
df_limits = df2_all_years[['CWNS_ID', 'value_received_year', 'Survey_parameter_mapping', 'standard_unit_desc', 'limit_value_standard_units']]

df_limits.shape

df_limits = df_limits.groupby(['CWNS_ID', 'value_received_year', 'Survey_parameter_mapping', 'standard_unit_desc'])['limit_value_standard_units'].mean().reset_index()

df_limits.shape

df_limits = df_limits.pivot(index=['CWNS_ID', 'value_received_year', 'standard_unit_desc'], 
                   columns='Survey_parameter_mapping', 
                   values='limit_value_standard_units').reset_index()

# add prefix DMR_limit_ to all columns from df_limits except CWNS_ID, value_received_year, standard_unit_desc
df_limits = df_limits.rename(columns={col: f"DL_{col}" for col in df_limits.columns if col not in ['CWNS_ID', 'value_received_year', 'standard_unit_desc']})
# remove rows without values in any of the DL limit columns
dl_columns = [col for col in df_limits.columns if col.startswith('DL_')]
df_limits = df_limits.dropna(subset=dl_columns, how='all').reset_index(drop=True)

df_limits.to_excel(os.path.join(path_out, 'DMR_limits_data.xlsx'), index=False)

(60, 5)
(55, 5)


In [13]:
# Merge DMR data with survey data

# Filter by study year of 2024
df_dmr = df_dmr[df_dmr['value_received_year'] == 2024].copy()

df_criteria_dmr = df_merged_criteria_npdes.merge(df_dmr, on='CWNS_ID', how='left')
df_criteria_dmr = df_criteria_dmr.merge(df_limits, on=['CWNS_ID', 'value_received_year', 'standard_unit_desc'], how='left')

df_criteria_dmr['DMR_standard_unit_desc'] = df_criteria_dmr['standard_unit_desc']
df_criteria_dmr.rename(columns={'standard_unit_desc': 'DL_standard_unit_desc'}, inplace=True)

# Unit harmonization

# Convert kg/day to mg/L
def convert_kgd_to_mgl(
    df: pd.DataFrame,
    columns_to_convert: list[str],
    unit_column: str = 'standard_unit_desc',
    flow_column: str = 'ACTUAL_FLOW_AVG',
    from_unit: str = 'kg/d',
    to_unit: str = 'mg/L',
    inplace: bool = False
) -> pd.DataFrame:
    """
    Convert selected columns from kg/day to mg/L using flow in MGD,
    and update the unit label for rows that were converted.

    Formula
    -------
    mg/L = (kg/day * 1e6 mg/kg) / (MGD * 3.78541e6 L/day)

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.
    columns_to_convert : list[str]
        Columns to convert.
    unit_column : str, default 'standard_unit_desc'
        Column containing the unit descriptor.
    flow_column : str, default 'ACTUAL_FLOW_AVG'
        Column containing flow in MGD.
    from_unit : str, default 'kg/d'
        Source unit to convert from.
    to_unit : str, default 'mg/L'
        Target unit to assign after conversion.
    inplace : bool, default False
        If True, modify the input DataFrame in place.

    Returns
    -------
    pd.DataFrame
        DataFrame with converted values and updated units.
    """
    result = df if inplace else df.copy()

    required_cols = [unit_column, flow_column]
    missing_required = [col for col in required_cols if col not in result.columns]
    if missing_required:
        raise KeyError(f"Missing required columns: {missing_required}")

    # Ensure numeric flow
    flow_mgd = pd.to_numeric(result[flow_column], errors='coerce')
    flow_l_per_day = flow_mgd * 3.78541e6

    # Only convert rows with the expected unit and valid nonzero flow
    converted_mask = (
        (result[unit_column] == from_unit) &
        flow_mgd.notna() &
        (flow_mgd != 0)
    )

    for col in columns_to_convert:
        if col not in result.columns:
            continue

        values = pd.to_numeric(result[col], errors='coerce')

        result.loc[converted_mask, col] = (
            (values.loc[converted_mask] * 1e6) / flow_l_per_day.loc[converted_mask]
        )

    # Update unit label only for rows that were actually converted
    result.loc[converted_mask, unit_column] = to_unit

    return result

df_criteria_dmr = convert_kgd_to_mgl(df_criteria_dmr,
                   columns_to_convert = ['DMR_BOD5_AVG', 'DMR_CBOD5_AVG', 'DMR_NO2_N_AVG', 'DMR_NO3_N_AVG', 'DMR_P_TOTAL_AVG', 'DMR_TN_AVG', 'DMR_TSS_AVG'],
                   unit_column = 'DMR_standard_unit_desc',
                   inplace=False)
df_criteria_dmr = convert_kgd_to_mgl(df_criteria_dmr,
                   columns_to_convert = ['DL_BOD5_AVG', 'DL_CBOD5_AVG', 'DL_NO2_N_AVG', 'DL_NO3_N_AVG', 'DL_P_TOTAL_AVG', 'DL_TN_AVG', 'DL_TSS_AVG'],
                   unit_column = 'DL_standard_unit_desc',
                   inplace=False)

# Convert percentage to mg/L
def convert_percent_reduction_columns_to_mgl(
    df: pd.DataFrame,
    percent_columns: list[str],
    prefix: str,
    unit_column: str = 'standard_unit_desc',
    from_unit: str = '%',
    to_unit: str = 'mg/L',
    inplace: bool = False,
    strict: bool = False
) -> pd.DataFrame:
    """
    Convert percent-reduction columns to mg/L using matching influent columns.

    The matching influent column is inferred by removing `prefix` from each
    percent column name.

    Example with prefix='DMR_':
        DMR_BOD5_AVG -> BOD5_AVG
        DMR_TSS_AVG  -> TSS_AVG

    Formula
    -------
    mg/L = (1 - percent / 100) * influent_value

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.
    percent_columns : list[str]
        Columns containing percent-reduction values to convert.
    prefix : str
        Prefix to remove from percent column names to infer influent columns.
    unit_column : str, default 'standard_unit_desc'
        Column containing the unit descriptor.
    from_unit : str, default '%'
        Unit indicating values are percentages.
    to_unit : str, default 'mg/L'
        Unit to assign after conversion.
    inplace : bool, default False
        If True, modify the input DataFrame in place.
    strict : bool, default False
        If True, raise an error when:
        - a percent column is missing
        - a percent column does not start with the prefix
        - an inferred influent column is missing

    Returns
    -------
    pd.DataFrame
        DataFrame with converted values and updated units.
    """
    result = df if inplace else df.copy()

    if unit_column not in result.columns:
        raise KeyError(f"Missing required column: {unit_column}")

    converted_any_mask = pd.Series(False, index=result.index)

    for percent_col in percent_columns:
        if percent_col not in result.columns:
            if strict:
                raise KeyError(f"Missing percent column: {percent_col}")
            continue

        if not percent_col.startswith(prefix):
            if strict:
                raise ValueError(
                    f"Percent column '{percent_col}' does not start with prefix '{prefix}'"
                )
            continue

        influent_col = percent_col[len(prefix):]

        if influent_col not in result.columns:
            if strict:
                raise KeyError(
                    f"Inferred influent column '{influent_col}' not found for '{percent_col}'"
                )
            continue

        percent_values = pd.to_numeric(result[percent_col], errors='coerce')
        influent_values = pd.to_numeric(result[influent_col], errors='coerce')

        mask = (
            (result[unit_column] == from_unit) &
            percent_values.notna() &
            influent_values.notna()
        )

        if not mask.any():
            continue

        result.loc[mask, percent_col] = (
            1.0 - (percent_values.loc[mask] / 100.0)
        ) * influent_values.loc[mask]

        converted_any_mask |= mask

    # Update unit for rows where at least one target column was converted
    result.loc[converted_any_mask, unit_column] = to_unit

    return result

df_criteria_dmr = convert_percent_reduction_columns_to_mgl(
     df=df_criteria_dmr,
     percent_columns=['DMR_BOD5_AVG', 'DMR_CBOD5_AVG', 'DMR_NO2_N_AVG', 'DMR_NO3_N_AVG', 'DMR_P_TOTAL_AVG', 'DMR_TN_AVG', 'DMR_TSS_AVG'],
     unit_column='DMR_standard_unit_desc',
     prefix='DMR_',
     strict = 'True'
)

df_criteria_dmr = convert_percent_reduction_columns_to_mgl(
     df=df_criteria_dmr,
     percent_columns=['DL_BOD5_AVG', 'DL_CBOD5_AVG', 'DL_NO2_N_AVG', 'DL_NO3_N_AVG', 'DL_P_TOTAL_AVG', 'DL_TN_AVG', 'DL_TSS_AVG'],
     unit_column='DL_standard_unit_desc',
     prefix='DL_',
     strict = 'True'
)


# Save final merged dataframe
df_criteria_dmr.to_excel(os.path.join(path_out, 'survey_criteria_DMR.xlsx'), index=False)

In [14]:
df_criteria_dmr

,CWNS_ID,FACILITY_ID,FACILITY_NAME,LATITUDE,LONGITUDE,CITY,STATE_CODE,AUTHORITY_NAME,COUNTY_NAME,DESIGN_FLOW,...,DMR_TN_AVG,DMR_TSS_AVG,DL_BOD5_AVG,DL_CBOD5_AVG,DL_NO2_N_AVG,DL_NO3_N_AVG,DL_P_TOTAL_AVG,DL_TN_AVG,DL_TSS_AVG,DMR_standard_unit_desc
0,17000721001,1113995.0,Stickney Treatment Plant,41.813900,-87.770400,Cicero,IL,MWRDGC,Cook,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,26000596001,1088610.0,GLWA WRRF,42.283200,-83.128500,Detroit,MI,DETROIT BOARD OF WATER CO,Wayne,2160,...,NaN,91.041667,85.0,NaN,NaN,NaN,NaN,NaN,85.000000,%
2,17000721009,1114001.0,Calumet Water Reclamation Plant,41.662500,-87.610000,Chicago,IL,MWRDGC,Cook,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11000001001,1142408.0,Blue Plains STP,38.816781,-77.032755,Washington,DC,D.C. WASA (BLUE PLAINS),District of Columbia,384,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,29001023002,1133278.0,Lemay WWTP,38.532700,-90.270900,Saint Louis,MO,Metropolitan St. Louis Sewer District,St. Louis,250,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,48000004001,1182787.0,Central WWTP - TRA,32.778700,-96.926400,Dallas,TX,Trinity River Authority,Dallas,189,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,29001023001,1133277.0,Bissell Point WWTP,38.673610,-90.194970,Saint Louis,MO,Metropolitan St. Louis Sewer District,St. Louis City,250,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,25000128001,1092422.0,MWRA Deer Island WWTF,42.350500,-70.956900,Boston,MA,Mass. Water Resources Authority,Suffolk,NaN,...,NaN,NaN,NaN,32.5,NaN,NaN,NaN,NaN,NaN,mg/L
8,17000721007,1113999.0,Terrence J O'Brien WRP,42.019100,-87.716300,Skokie,IL,MWRDGC,Cook,450,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,6004010001,1175491.0,Hyperion WRP,33.930800,-118.434900,Playa Del Rey,CA,"Los Angeles, City of - Bureau of Sanitation",Los Angeles,450,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
import numpy as np
import pandas as pd
import re

# Next step is to identify data on effluent concentration, for only those CWNS IDs 
# and parameters with influent data available. 
# Take DMR effluent data if available (column exists AND row has value), 
# if not take NPDES limit data if available (column exists AND row has value).

param_cols = [
    ['BOD5_AVG',         'DMR_BOD5_AVG',       'NPDES2_BOD5_AVG_mg/L'],
    ['CBOD5_AVG',        'DMR_CBOD5_AVG',       'NPDES2_CBOD5_AVG_mg/L'],
    ['SS_AVG',           '',                    ''],
    ['VSS_AVG',          '',                    ''],
    ['TS_AVG',           '',                    ''],
    ['TSS_AVG',          'DMR_TSS_AVG',         'NPDES2_TSS_AVG_mg/L'],
    ['VTS_AVG',          '',                    ''],
    ['TKN_AVG',          '',                    ''],
    ['TN_AVG',           'DMR_TN_AVG',          'NPDES_total nitrogen_NPDES_final'],
    ['NH3_N_AVG',        '',                    'NPDES_ammonia_NPDES_final'],
    ['NO3_N_AVG',        'DMR_NO3_N_AVG',       'NPDES_nitrate'],
    ['NO2_N_AVG',        'DMR_NO2_N_AVG',       ''],
    ['NO3_N+NO2_N_AVG',  '',                    'NPDES_nitrates+nitrites'],
    ['P_TOTAL_AVG',      'DMR_P_TOTAL_AVG',     'NPDES_total phosphorus_NPDES_final'],
    ['P_SOLID_AVG',      '',                    ''],
]

# Table mapping influent parameters to corresponding NPDES and DMR parameter columns.
map_data = pd.DataFrame(
    [(x[0], x[1], x[2]) for x in param_cols],
    columns=['Influent_param_col', 'DMR_param_col', 'NPDES_param_col']
)

# Replace empty strings with NaN so we can use pd.notna() checks reliably.
map_data = map_data.replace('', pd.NA)

for influent, dmr, npdes in map_data.itertuples(index=False):
    print(f"Processing influent parameter '{influent}' with DMR column '{dmr}' and NPDES column '{npdes}'")
    
    # Check if influent column exists
    if influent not in df_criteria_dmr.columns:
        print(f"  WARNING: Influent column '{influent}' not found in DataFrame. Skipping.")
        continue
    
    # Mask for rows where the influent parameter has a value (not null)
    mask = df_criteria_dmr[influent].notna()
    
    # Create effluent columns - initialize with None for object dtype
    effluent_col      = influent + '_effluent'
    effluent_col_unit = influent + '_effluent_units'
    df_criteria_dmr[effluent_col]      = None
    df_criteria_dmr[effluent_col_unit] = None
    
    # Row-level flag to track which rows received DMR data
    dmr_data_found_flg = pd.Series(0, index=df_criteria_dmr.index, dtype=int)
    
    # -----------------------------------------------------------------------
    # Step 1: Try to use DMR data if available
    # -----------------------------------------------------------------------
    dmr_filled_count = 0
    if pd.notna(dmr) and dmr in df_criteria_dmr.columns:
        # Row-level mask: influent has value AND DMR has a value in this specific row
        dmr_row_mask = mask & df_criteria_dmr[dmr].notna()
        dmr_filled_count = dmr_row_mask.sum()
        
        if dmr_filled_count > 0:
            # Write DMR value to effluent column for matched rows
            df_criteria_dmr.loc[dmr_row_mask, effluent_col] = \
                df_criteria_dmr.loc[dmr_row_mask, dmr].values
            
            # Write DMR unit if column exists
            if 'DMR_standard_unit_desc' in df_criteria_dmr.columns:
                df_criteria_dmr.loc[dmr_row_mask, effluent_col_unit] = \
                    df_criteria_dmr.loc[dmr_row_mask, 'DMR_standard_unit_desc'].values
            
            # Set flag to 1 for rows where DMR data was successfully written
            dmr_data_found_flg.loc[dmr_row_mask] = 1
        
        print(f"  DMR: {dmr_filled_count} rows filled")
    
    # -----------------------------------------------------------------------
    # Step 2: Try to use NPDES data as fallback where DMR wasn't used
    # -----------------------------------------------------------------------
    npdes_filled_count = 0
    if pd.notna(npdes) and npdes in df_criteria_dmr.columns:
        # Row-level mask: influent has value AND DMR flag NOT set AND NPDES has a value
        npdes_row_mask = mask & (dmr_data_found_flg == 0) & df_criteria_dmr[npdes].notna()
        npdes_filled_count = npdes_row_mask.sum()
        
        if npdes_filled_count > 0:
            # Write NPDES value to effluent column for matched rows
            df_criteria_dmr.loc[npdes_row_mask, effluent_col] = \
                df_criteria_dmr.loc[npdes_row_mask, npdes].values
            
            # NPDES columns are standardized in mg/L
            df_criteria_dmr.loc[npdes_row_mask, effluent_col_unit] = 'mg/L'
        
        print(f"  NPDES: {npdes_filled_count} rows filled")
    
    # Summary for this parameter
    total_filled = df_criteria_dmr[effluent_col].notna().sum()
    print(f"  Total: {total_filled} rows have effluent data\n")

# ---------------------------------------------------------------------------
# Calculate pollutant reductions between influent and effluent 
# ---------------------------------------------------------------------------

calculate_criteria_reductions = True
if calculate_criteria_reductions:    
    for influent_col, criteria_names in pollutants_dict.items():
        effluent_col = influent_col + "_effluent"
        reduction_col = influent_col + "_reduction" 
        # Convert columns to numeric, coercing errors to NaN
        df_criteria_dmr[influent_col] = pd.to_numeric(df_criteria_dmr[influent_col], errors='coerce')
        df_criteria_dmr[effluent_col] = pd.to_numeric(df_criteria_dmr[effluent_col], errors='coerce')
        # Perform subtraction and create the new column
        df_criteria_dmr[reduction_col] = df_criteria_dmr[influent_col] - df_criteria_dmr[effluent_col]

# ---------------------------------------------------------------------------
# Write one Excel tab per influent parameter with relevant columns
# ---------------------------------------------------------------------------
facility_bio_cols = [
    'CWNS_ID',
    'Assigned TT'
]

def safe_sheet_name(name, max_len=31):
    """Sanitize a string to be a valid Excel sheet name (max 31 chars, no special chars)."""
    name = str(name)
    name = re.sub(r'[:\\/*?\[\]]', '_', name)  # Replace invalid Excel characters
    return name[:max_len]

file_path = os.path.join(path_out, 'survey_effluent_repo.xlsx')

# Write all parameter tabs into a single Excel workbook
with pd.ExcelWriter(file_path, engine='openpyxl', mode='w') as writer:
    for influent, dmr, npdes in map_data.itertuples(index=False):
        sheet_name = safe_sheet_name(influent)

        # Build list of columns to include for this parameter tab
        cols_to_include = (
            facility_bio_cols +
            [influent] +
            [influent + '_UNITS'] +
            [influent + '_effluent'] +
            [influent + '_effluent_units'] +
            [influent + "_reduction"]
        )

        # Correct any column names that don't exist by trying alternate naming convention
        corrected_cols = []
        for col in cols_to_include:
            if col not in df_criteria_dmr.columns:
                # Try alternate: strip _AVG from UNITS column name
                col = col.replace('_AVG_UNITS', '') + '_UNITS'
            corrected_cols.append(col)
        cols_to_include = corrected_cols

        # Keep only rows where influent OR effluent has data
        df_criteria_dmr_filtered = df_criteria_dmr[
            df_criteria_dmr[influent].notna() |
            df_criteria_dmr[influent + '_effluent'].notna()
        ]

        # Drop fully duplicate rows across the selected columns
        df_criteria_dmr_filtered = df_criteria_dmr_filtered.drop_duplicates(subset=cols_to_include)

        # Write this parameter's data to its own sheet tab
        df_criteria_dmr_filtered[cols_to_include].to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"Tab '{sheet_name}' written successfully")


# ---------------------------------------------------------------------------
# Write all at once to an xlsx file
# ---------------------------------------------------------------------------
file_path = os.path.join(path_out, 'survey_effluent_one.xlsx')
df_criteria_dmr.to_excel(file_path, index=False)

print("\nProcessing complete!")

Processing influent parameter 'BOD5_AVG' with DMR column 'DMR_BOD5_AVG' and NPDES column 'NPDES2_BOD5_AVG_mg/L'
  DMR: 0 rows filled
  NPDES: 21 rows filled
  Total: 21 rows have effluent data

Processing influent parameter 'CBOD5_AVG' with DMR column 'DMR_CBOD5_AVG' and NPDES column 'NPDES2_CBOD5_AVG_mg/L'
  DMR: 1 rows filled
  NPDES: 0 rows filled
  Total: 1 rows have effluent data

Processing influent parameter 'SS_AVG' with DMR column 'nan' and NPDES column 'nan'
  Total: 0 rows have effluent data

Processing influent parameter 'VSS_AVG' with DMR column 'nan' and NPDES column 'nan'
  Total: 0 rows have effluent data

Processing influent parameter 'TS_AVG' with DMR column 'nan' and NPDES column 'nan'
  Total: 0 rows have effluent data

Processing influent parameter 'TSS_AVG' with DMR column 'DMR_TSS_AVG' and NPDES column 'NPDES2_TSS_AVG_mg/L'
  DMR: 3 rows filled
  NPDES: 18 rows filled
  Total: 21 rows have effluent data

Processing influent parameter 'VTS_AVG' with DMR column 'na

In [ ]:
# Summarize count, min-max, avg, median, SD, and plot density distributions for numerical columns

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

def summarize_and_plot(df, output_folder):
    """
    Summarize count of values for each column and create density plots for numerical columns.
    Zeros are excluded from numerical statistics as they represent unmeasured values.
    
    Parameters:
    - df: pandas DataFrame
    - output_folder: path to folder where plots will be saved
    """
    
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # --- Part 1: Summarize count of values for each column ---
    print("=" * 50)
    print("VALUE COUNTS SUMMARY FOR EACH COLUMN")
    print("=" * 50)
    
    # Get numerical columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    for col in df.columns:
        print(f"\n--- {col} ---")
        print(f"Total count: {df[col].count()}")
        print(f"Null/NA count: {df[col].isna().sum()}")
        print(f"Unique values: {df[col].nunique()}")
        
        # Print min, max, average for numerical columns (excluding zeros and NAs)
        if col in numerical_cols:
            # Filter out NAs and zeros
            clean_data = df[col].dropna()
            clean_data = clean_data[clean_data != 0]
            
            zero_count = (df[col] == 0).sum()
            print(f"Zero count (excluded): {zero_count}")
            print(f"Valid count (non-zero, non-NA): {len(clean_data)}")
            
            if len(clean_data) > 0:
                print(f"Min: {clean_data.min()}")
                print(f"Max: {clean_data.max()}")
                print(f"Average: {clean_data.mean():.4f}")
                print(f"Median: {clean_data.median()}")
                print(f"Std Dev: {clean_data.std():.4f}")
            else:
                print("No valid data to calculate stats")
        
        print(df[col].value_counts().head(10))  # Show top 10 values
    
    # --- Part 2: Summary table for numerical columns ---
    print("\n" + "=" * 50)
    print("NUMERICAL COLUMNS SUMMARY TABLE (Zeros Excluded)")
    print("=" * 50)
    
    # Create summary dataframe for numerical columns
    summary_data = []
    for col in numerical_cols:
        # Filter out NAs and zeros
        clean_data = df[col].dropna()
        clean_data = clean_data[clean_data != 0]
        
        summary_data.append({
            'Column': col,
            'Total Count': df[col].count(),
            'Zero Count': (df[col] == 0).sum(),
            'Missing': df[col].isna().sum(),
            'Valid Count': len(clean_data),
            'Min': clean_data.min() if len(clean_data) > 0 else np.nan,
            'Max': clean_data.max() if len(clean_data) > 0 else np.nan,
            'Average': clean_data.mean() if len(clean_data) > 0 else np.nan,
            'Median': clean_data.median() if len(clean_data) > 0 else np.nan,
            'Std Dev': clean_data.std() if len(clean_data) > 0 else np.nan
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    
    # --- Part 3: Density plots for numerical columns ---
    print("\n" + "=" * 50)
    print("CREATING DENSITY PLOTS FOR NUMERICAL COLUMNS")
    print("=" * 50)
    
    for col in numerical_cols:
        # Filter out blanks, NAs, and zeros
        clean_data = df[col].dropna()
        clean_data = clean_data[clean_data != 0]
        
        # Skip if no data left after filtering
        if len(clean_data) == 0:
            print(f"Skipping {col}: No valid data after filtering")
            continue
        
        # Skip if only one unique value (can't compute density)
        if clean_data.nunique() < 2:
            print(f"Skipping {col}: Only {clean_data.nunique()} unique value(s) - cannot compute density")
            continue
        
        # Skip if standard deviation is zero (all values are the same)
        if clean_data.std() == 0:
            print(f"Skipping {col}: Zero variance - cannot compute density")
            continue
        
        # Create density plot with error handling
        try:
            plt.figure(figsize=(8, 6))
            clean_data.plot(kind='density', color='steelblue', linewidth=2)
            plt.title(f'Density Distribution: {col}\n(Zeros and NAs excluded)')
            plt.xlabel(col)
            plt.ylabel('Density')
            plt.grid(True, alpha=0.3)
            
            # Save plot
            # Clean column name for filename (remove special characters)
            safe_col_name = "".join(c if c.isalnum() or c in ('_', '-') else '_' for c in col)
            filename = f"{safe_col_name}_density_dist.png"
            filepath = os.path.join(output_folder, filename)
            plt.savefig(filepath, dpi=150, bbox_inches='tight')
            plt.close()
            
            print(f"Saved: {filepath}")
            
        except Exception as e:
            plt.close()
            print(f"Skipping {col}: Error creating plot - {str(e)}")
    
    print("\nDone!")
    
    # Return summary dataframe for further use if needed
    return summary_df

summary = summarize_and_plot(df_merged_criteria, path_out_plots + '/density_plots')
summary.to_excel(os.path.join(path_out, 'numerical_columns_summary.xlsx'), index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

def create_versatile_plots(df, columns_to_plot=None, y_axis_labels=None, x_axis_column='CWNS_ID',
                           x_axis_label=None, plot_title=None, include_statistics=True,
                           plot_type='boxplot', stats_display='smart', output_dir=None,
                           figsize=None, display_in_notebook=True, save_files=True,
                           palette="Set2", max_categories=20, boxplot_colors=None,
                           column_colors=None, single_color=None, horizontal=False,
                           sort_order=None, decimal_places=2, vlines_config=None, 
                           group_by_column=None, group_sort_order=None, 
                           group_label_text=None, font_scale=1.0, bold_axis_labels=True,
                           bold_tick_labels=True, bold_title=True, bold_group_labels=True,
                           bold_value_labels=True, value_label_offset=None):
    """
    Versatile plotting function that can create boxplots or column plots with custom colors
    
    Parameters:
    -----------
    decimal_places : int or dict
        If int, use same decimal places for all columns
        If dict, use column-specific decimal places {column_name: decimal_places}
    vlines_config : dict
        Dictionary mapping column names to their vertical line configurations.
        Each configuration is a dict with 'names', 'values', and 'colors' keys.
    group_by_column : str
        Column name to group data by. Data within each group will be sorted.
        Groups will be displayed as secondary y-axis labels with divider lines.
    group_sort_order : str
        Sort order within groups: 'asc' or 'desc' (applied to the grouping column values)
    group_label_text : str
        Text label for the secondary y-axis (group labels).
        Default is the group_by_column name.
    font_scale : float
        Multiplier for all font sizes. Default is 1.0.
        Use 1.2 for 20% larger, 1.5 for 50% larger, 2.0 for double size, etc.
    bold_axis_labels : bool
        Whether to make axis labels (x-axis and y-axis labels) bold. Default is True.
    bold_tick_labels : bool
        Whether to make tick labels bold. Default is True.
    bold_title : bool
        Whether to make the title bold. Default is True.
    bold_group_labels : bool
        Whether to make secondary y-axis (group) labels bold. Default is True.
    bold_value_labels : bool
        Whether to make value labels (data labels on bars) bold. Default is True.
    value_label_offset : float or None
        Horizontal offset for value labels to prevent overlap with secondary y-axis.
        If None, automatically calculated based on whether group_by_column is used.
        Positive values move labels to the right, negative to the left.
    """
    
    # Validate vlines_config if provided
    if vlines_config is not None:
        if not isinstance(vlines_config, dict):
            raise ValueError("vlines_config must be a dictionary")
        
        for col_name, vline_def in vlines_config.items():
            if not isinstance(vline_def, dict):
                raise ValueError(f"vlines_config['{col_name}'] must be a dictionary")
            
            required_keys = {'names', 'values', 'colors'}
            if not required_keys.issubset(vline_def.keys()):
                raise ValueError(f"vlines_config['{col_name}'] must contain keys: {required_keys}")
            
            names = vline_def['names']
            values = vline_def['values']
            colors = vline_def['colors']
            
            if not (isinstance(names, list) and isinstance(values, list) and isinstance(colors, list)):
                raise ValueError(f"vlines_config['{col_name}'] values must be lists")
            
            if not (len(names) == len(values) == len(colors)):
                raise ValueError(f"vlines_config['{col_name}'] - names, values, and colors must have the same length")
    
    # Validate grouping parameters
    if group_by_column is not None:
        if group_by_column not in df.columns:
            raise ValueError(f"group_by_column '{group_by_column}' not found in dataframe")
        if group_sort_order is not None and group_sort_order not in ['asc', 'desc']:
            raise ValueError("group_sort_order must be 'asc' or 'desc'")
    
    # Validate font_scale
    if font_scale <= 0:
        raise ValueError("font_scale must be a positive number")
    
    # Set default group_label_text if not provided
    if group_label_text is None and group_by_column is not None:
        group_label_text = group_by_column
    
    # Set default output directory based on plot type
    if output_dir is None:
        base_dir = 'plots'
        orientation_suffix = '_horizontal' if horizontal else ''
        if plot_type == 'boxplot':
            output_dir = os.path.join(base_dir, f'boxplots_{stats_display}{orientation_suffix}')
        else:
            output_dir = os.path.join(base_dir, f'column_plots_{stats_display}{orientation_suffix}')
    
    # Create output directory if it doesn't exist
    if save_files and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Determine which columns to plot
    if columns_to_plot is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        numeric_cols = [col for col in numeric_cols if col != x_axis_column]
    else:
        numeric_cols = [col for col in columns_to_plot if col in df.columns and pd.api.types.is_numeric_dtype(df[col])]
    
    # Process y-axis labels
    if y_axis_labels is None:
        label_mapping = {col: col for col in numeric_cols}
    elif isinstance(y_axis_labels, list):
        label_mapping = dict(zip(numeric_cols, y_axis_labels))
    elif isinstance(y_axis_labels, dict):
        label_mapping = y_axis_labels
    else:
        raise ValueError("y_axis_labels must be a list or dictionary")
    
    # Determine x-axis display label
    x_display_label = x_axis_label if x_axis_label is not None else x_axis_column
    
    print(f"Creating {'horizontal' if horizontal else 'vertical'} {plot_type}s with {stats_display} statistics for {len(numeric_cols)} variables")
    print(f"Font scale: {font_scale}x")
    
    for i, col in enumerate(numeric_cols, 1):
        print(f"Processing {i}/{len(numeric_cols)}: {col}")
        
        # Prepare data for plotting - FILTER OUT NA/ZERO VALUES FIRST
        plot_data = df[[x_axis_column, col]].copy()
        
        # Add grouping column if specified
        if group_by_column is not None:
            plot_data[group_by_column] = df[group_by_column]
        
        # Remove rows where the column value is NA or zero
        plot_data = plot_data[plot_data[col].notna()]
        if plot_type == 'column':
            # For column plots, also remove zero values
            plot_data = plot_data[plot_data[col] != 0]
        
        if len(plot_data) == 0:
            print(f"  Skipping {col} - no data available")
            continue
        
        # Count actual data points after filtering
        num_data_points = len(plot_data)
        print(f"  Number of data points: {num_data_points}")
        
        # Convert x_axis_column to string
        plot_data[x_axis_column] = plot_data[x_axis_column].astype(str)
        
        # Apply grouping and sorting if specified
        group_positions = {}  # Store group boundaries for divider lines
        group_labels_dict = {}  # Store group label positions
        
        if group_by_column is not None:
            # Sort by group column first
            ascending_group = group_sort_order == 'asc' if group_sort_order else True
            plot_data = plot_data.sort_values(by=group_by_column, ascending=ascending_group)
            
            # Within each group, sort by the data value in descending order
            plot_data = plot_data.sort_values(by=[group_by_column, col], 
                                             ascending=[ascending_group, False])
            
            # Reset index for proper positioning
            plot_data = plot_data.reset_index(drop=True)
            
            # Calculate group boundaries and label positions
            current_pos = 0
            for group_val in plot_data[group_by_column].unique():
                group_data_indices = plot_data[plot_data[group_by_column] == group_val].index
                group_size = len(group_data_indices)
                
                # Store group boundaries (for divider lines)
                group_positions[group_val] = (current_pos, current_pos + group_size)
                
                # Store group label position (middle of the group)
                group_labels_dict[group_val] = current_pos + (group_size - 1) / 2.0
                
                current_pos += group_size
            
            # Store boundaries between groups for divider lines
            group_boundaries = [pos[1] - 0.5 for pos in list(group_positions.values())[:-1]]
            
            print(f"  Data grouped by '{group_by_column}' and sorted within groups")
            print(f"  Group boundaries at positions: {group_boundaries}")
        else:
            # Calculate number of unique categories WITH DATA
            num_categories_with_data = len(plot_data[x_axis_column].unique())
            
            # Apply sorting if specified
            if sort_order:
                if sort_order in ['asc', 'desc']:
                    plot_data = plot_data.sort_values(by=x_axis_column, ascending=(sort_order == 'asc'))
                elif sort_order in ['numeric_asc', 'numeric_desc']:
                    plot_data = plot_data.sort_values(by=x_axis_column, 
                                                      key=lambda x: pd.to_numeric(x, errors='coerce'),
                                                      ascending=(sort_order == 'numeric_asc'))
                elif sort_order in ['data_asc', 'data_desc']:
                    value_counts = plot_data[x_axis_column].value_counts()
                    plot_data = plot_data.sort_values(by=x_axis_column, 
                                                      key=lambda x: value_counts[x],
                                                      ascending=(sort_order == 'data_asc'))
                elif sort_order in ['value_asc', 'value_desc']:
                    plot_data = plot_data.sort_values(by=col, ascending=(sort_order == 'value_asc'))
        
        # Calculate number of unique categories WITH DATA
        num_categories_with_data = len(plot_data[x_axis_column].unique())
        
        # Calculate optimal figsize based on ACTUAL DATA POINTS
        if figsize is None:
            calculated_figsize = calculate_optimal_figsize(
                num_categories=num_categories_with_data,
                num_data_points=num_data_points, 
                horizontal=horizontal,
                plot_type=plot_type
            )
        else:
            calculated_figsize = figsize
        
        print(f"  Figure size: {calculated_figsize}, Categories with data: {num_categories_with_data}, Data points: {num_data_points}")
        
        # Create the main plot with CALCULATED figsize
        fig, ax = plt.subplots(figsize=calculated_figsize)
        
        # Calculate font sizes based on number of data points and font_scale
        base_fonts = calculate_base_font_sizes(num_data_points, font_scale)
        
        if plot_type == 'boxplot':
            if horizontal:
                sns.boxplot(data=plot_data, y=x_axis_column, x=col, ax=ax, orient='h')
                sns.stripplot(data=plot_data, y=x_axis_column, x=col, ax=ax, color='red', alpha=0.5, orient='h')
            else:
                sns.boxplot(data=plot_data, x=x_axis_column, y=col, ax=ax)
                sns.stripplot(data=plot_data, x=x_axis_column, y=col, ax=ax, color='red', alpha=0.5)
        elif plot_type == 'column':
            if horizontal:
                # Use original x_axis_column for bar labels
                sns.barplot(data=plot_data, y=x_axis_column, x=col, ax=ax, orient='h', 
                            color=single_color or 'steelblue', order=plot_data[x_axis_column])
            else:
                sns.barplot(data=plot_data, x=x_axis_column, y=col, ax=ax, 
                            color=single_color or 'steelblue')
        
        # Add divider lines between groups if grouping is applied
        if group_by_column is not None and group_boundaries:
            for boundary_pos in group_boundaries:
                if horizontal:
                    # For horizontal plots, draw horizontal divider lines
                    ax.axhline(y=boundary_pos, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
                else:
                    # For vertical plots, draw vertical divider lines
                    ax.axvline(x=boundary_pos, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
        
        # Add group labels on secondary y-axis (only once per group)
        if group_by_column is not None and group_labels_dict:
            if horizontal:
                # Create secondary y-axis for group labels
                ax2 = ax.twinx()
                ax2.set_ylim(ax.get_ylim())
                
                # Set group labels at their positions
                group_label_positions = list(group_labels_dict.values())
                group_label_names = list(group_labels_dict.keys())
                
                group_label_weight = 'bold' if bold_group_labels else 'normal'
                group_axis_label_weight = 'bold' if bold_axis_labels else 'normal'
                
                ax2.set_yticks(group_label_positions)
                ax2.set_yticklabels(group_label_names, fontsize=base_fonts['group_label'], 
                                   fontweight=group_label_weight, color='darkblue')
                # Use the provided group_label_text or default to group_by_column
                ax2.set_ylabel(group_label_text, fontsize=base_fonts['group_axis_label'], 
                             fontweight=group_axis_label_weight, color='darkblue')
                ax2.tick_params(axis='y', labelcolor='darkblue')
        
        # Add vertical lines if specified for this column
        has_vlines = False
        if vlines_config is not None and col in vlines_config:
            vline_def = vlines_config[col]
            vline_names = vline_def['names']
            vline_values = vline_def['values']
            vline_colors = vline_def['colors']
            
            for vline_name, vline_value, vline_color in zip(vline_names, vline_values, vline_colors):
                if horizontal:
                    # For horizontal plots, vertical lines are drawn on x-axis
                    ax.axvline(x=vline_value, color=vline_color, linestyle='--', linewidth=2.5, 
                              label=vline_name, alpha=0.8)
                else:
                    # For vertical plots, vertical lines are drawn on y-axis
                    ax.axhline(y=vline_value, color=vline_color, linestyle='--', linewidth=2.5, 
                              label=vline_name, alpha=0.8)
            
            has_vlines = True
            print(f"  Added {len(vline_values)} vertical line(s) for {col}")
        
        # Add legend for vertical lines if they exist
        if has_vlines:
            ax.legend(loc='best', fontsize=base_fonts['legend'], framealpha=0.95)
        
        # Set labels and title with LARGER font sizes
        axis_label_weight = 'bold' if bold_axis_labels else 'normal'
        title_weight = 'bold' if bold_title else 'normal'
        
        if horizontal:
            ax.set_xlabel(label_mapping.get(col, col), fontsize=base_fonts['axis_label'], fontweight=axis_label_weight)
            ax.set_ylabel(x_display_label, fontsize=base_fonts['axis_label'], fontweight=axis_label_weight)
        else:
            ax.set_ylabel(label_mapping.get(col, col), fontsize=base_fonts['axis_label'], fontweight=axis_label_weight)
            ax.set_xlabel(x_display_label, fontsize=base_fonts['axis_label'], fontweight=axis_label_weight)
        
        # Set tick labels with LARGER fonts
        tick_label_weight = 'semibold' if bold_tick_labels else 'normal'
        ax.tick_params(axis='both', labelsize=base_fonts['tick_label'])
        
        # For horizontal plots, make y-axis (category) labels bold or normal
        if horizontal:
            ax.tick_params(axis='y', labelsize=base_fonts['tick_label'], width=1.5)
            # Make y-axis labels bold or normal based on parameter
            for label in ax.get_yticklabels():
                label.set_fontweight(tick_label_weight)
        
        if isinstance(plot_title, str):
            ax.set_title(plot_title, fontsize=base_fonts['title'], fontweight=title_weight)
        elif isinstance(plot_title, dict):
            ax.set_title(plot_title.get(col, label_mapping.get(col, col)), fontsize=base_fonts['title'], fontweight=title_weight)
        elif plot_title is None:
            ax.set_title(label_mapping.get(col, col), fontsize=base_fonts['title'], fontweight=title_weight)
        
        # Determine decimal places for this column
        if isinstance(decimal_places, dict):
            col_decimal_places = decimal_places.get(col, 2)  # Default to 2 if not specified
        else:
            col_decimal_places = decimal_places
        
        # Calculate automatic offset for value labels if grouping is applied
        auto_offset = None
        if horizontal and group_by_column is not None and value_label_offset is None:
            # Get the maximum value to calculate appropriate offset
            max_value = plot_data[col].max()
            # Offset is 2% of the max value to prevent overlap with secondary y-axis
            auto_offset = max_value * 0.02
        
        # Use provided offset or auto-calculated offset
        final_offset = value_label_offset if value_label_offset is not None else auto_offset
        
        # Add value labels for column plots with CORRECT DECIMAL PLACES
        if plot_type == 'column':
            value_label_weight = 'semibold' if bold_value_labels else 'normal'
            for p in ax.patches:
                value = p.get_width() if horizontal else p.get_height()
                
                # Format the value with the correct number of decimal places
                if col_decimal_places == 0:
                    formatted_value = f'{value:.0f}'
                else:
                    formatted_value = f'{value:.{col_decimal_places}f}'
                
                if horizontal:
                    # Apply offset to prevent overlap with secondary y-axis
                    x_pos = p.get_width() + (final_offset if final_offset else 0)
                    ax.text(x_pos, p.get_y() + p.get_height()/2., 
                            formatted_value, ha='left', va='center', 
                            fontsize=base_fonts['value_label'], fontweight=value_label_weight)
                else:
                    ax.text(p.get_x() + p.get_width()/2., p.get_height(), 
                            formatted_value, ha='center', va='bottom',
                            fontsize=base_fonts['value_label'], fontweight=value_label_weight)
        
        # Adjust layout with better margins
        if horizontal:
            # Calculate maximum label length
            max_label_length = max([len(str(label)) for label in plot_data[x_axis_column].unique()])
            
            # More space for larger fonts
            if max_label_length > 50:
                left_margin = 0.40
            elif max_label_length > 40:
                left_margin = 0.35
            elif max_label_length > 30:
                left_margin = 0.30
            else:
                left_margin = 0.25
            
            # Also add right margin for value labels and secondary y-axis
            # Increase right margin if we have value labels with offset
            right_margin = 0.90 if (group_by_column is not None and plot_type == 'column') else 0.88 if group_by_column is not None else 0.82
            
            fig.subplots_adjust(left=left_margin, right=right_margin)
        
        # Use tight_layout with padding
        plt.tight_layout(pad=2.0)
        
        # Save the plot if requested
        if save_files:
            filename = f"{col.replace('/', '_').replace(' ', '_')}_{plot_type}{'_horizontal' if horizontal else ''}.png"
            filepath = os.path.join(output_dir, filename)
            plt.savefig(filepath, dpi=300, bbox_inches='tight', pad_inches=0.3)
            print(f"  Saved: {filename}")
        
        # Display in notebook if requested
        if display_in_notebook:
            plt.show()
        else:
            plt.close()

    if save_files:
        print(f"\nCompleted! All plots saved in '{output_dir}' directory")


def calculate_optimal_figsize(num_categories, num_data_points, horizontal=False, plot_type='boxplot'):
    """
    Calculate optimal figure size based on ACTUAL number of data points and categories.
    Increased sizes to accommodate larger fonts.
    
    Parameters:
    -----------
    num_categories : int
        Number of unique categories WITH DATA
    num_data_points : int
        Number of actual data points (non-NA, non-zero values)
    horizontal : bool
        Whether the plot is horizontal
    plot_type : str
        Type of plot ('boxplot' or 'column')
    
    Returns:
    --------
    tuple : (width, height) for figsize
    """
    
    if horizontal:
        # For horizontal plots, height depends on number of data points
        # Increased bar height to accommodate larger fonts
        bar_height = 0.5  # Increased from 0.4
        base_height = 3.5  # Increased from 2.5
        
        # Calculate height based on actual data points
        optimal_height = base_height + (num_data_points * bar_height)
        
        # Ensure minimum and maximum heights
        optimal_height = max(5, min(30, optimal_height))
        
        # Increased width for larger fonts
        if num_data_points <= 5:
            optimal_width = 12  # Increased from 10
        elif num_data_points <= 10:
            optimal_width = 13  # Increased from 11
        elif num_data_points <= 20:
            optimal_width = 14  # Increased from 12
        else:
            optimal_width = 16  # Increased from 14
            
    else:
        # For vertical plots, width depends on number of data points
        bar_width = 0.6  # Increased from 0.5
        base_width = 6  # Increased from 5
        
        # Calculate width based on actual data points
        optimal_width = base_width + (num_data_points * bar_width)
        
        # Ensure minimum and maximum widths
        optimal_width = max(10, min(30, optimal_width))
        
        # Height with more space for larger fonts
        if num_data_points <= 5:
            optimal_height = 8
        elif num_data_points <= 10:
            optimal_height = 9
        elif num_data_points <= 20:
            optimal_height = 10
        else:
            optimal_height = 12
    
    return (optimal_width, optimal_height)


def calculate_base_font_sizes(num_data_points, font_scale=1.0):
    """
    Calculate base font sizes based on number of data points.
    INTELLIGENTLY SCALES FONTS to prevent oversizing with low data point counts.
    
    Parameters:
    -----------
    num_data_points : int
        Number of actual data points
    font_scale : float
        Multiplier for all font sizes (default 1.0)
    
    Returns:
    --------
    dict : Dictionary of font sizes for different elements
    """
    # Use a scaling factor that reduces font sizes more gradually for small datasets
    # This prevents text from being too large when there are only a few data points
    
    if num_data_points <= 2:
        # Very small datasets - use moderate font sizes
        base_sizes = {
            'title': 16,
            'axis_label': 14,
            'tick_label': 12,
            'value_label': 11,
            'legend': 11,
            'group_label': 11,
            'group_axis_label': 12
        }
    elif num_data_points <= 3:
        base_sizes = {
            'title': 18,
            'axis_label': 16,
            'tick_label': 14,
            'value_label': 12,
            'legend': 12,
            'group_label': 12,
            'group_axis_label': 14
        }
    elif num_data_points <= 5:
        base_sizes = {
            'title': 19,
            'axis_label': 17,
            'tick_label': 15,
            'value_label': 13,
            'legend': 13,
            'group_label': 13,
            'group_axis_label': 15
        }
    elif num_data_points <= 10:
        base_sizes = {
            'title': 19,
            'axis_label': 17,
            'tick_label': 15,
            'value_label': 13,
            'legend': 13,
            'group_label': 13,
            'group_axis_label': 15
        }
    elif num_data_points <= 20:
        base_sizes = {
            'title': 19,
            'axis_label': 17,
            'tick_label': 15,
            'value_label': 13,
            'legend': 13,
            'group_label': 13,
            'group_axis_label': 15
        }
    elif num_data_points <= 30:
        base_sizes = {
            'title': 19,
            'axis_label': 17,
            'tick_label': 15,
            'value_label': 13,
            'legend': 13,
            'group_label': 13,
            'group_axis_label': 15
        }
    else:
        base_sizes = {
            'title': 19,
            'axis_label': 17,
            'tick_label': 15,
            'value_label': 13,
            'legend': 13,
            'group_label': 13,
            'group_axis_label': 15
        }
    
    # Apply font_scale multiplier to all sizes
    scaled_sizes = {key: int(value * font_scale) for key, value in base_sizes.items()}
    
    return scaled_sizes


entity_to_label = {
    'DESIGN_FLOW': 'Design Flow (MGD)',
    'ACTUAL_FLOW_AVG': 'Actual Flow Avg (MGD)',
    'BOD5_AVG': 'BOD5 Avg (mg/L)',
    'CBOD5_AVG': 'CBOD5 Avg (mg/L)',
    'SS_AVG': 'SS Avg (mg/L)',
    'VSS_AVG': 'VSS Avg (mg/L)',
    'TS_AVG': 'TS Avg (mg/L)',
    'TSS_AVG': 'TSS Avg (mg/L)',
    'VTS_AVG': 'VTS Avg (mg/L)',
    'TKN_AVG': 'TKN Avg (mg/L)',
    'TN_AVG': 'TN Avg (mg/L)',
    'NH3_N_AVG': 'NH3-N Avg (mg/L)',
    'NO3_N_AVG': 'NO3-N Avg (mg/L)',
    'NO2_N_AVG': 'NO2-N Avg (mg/L)',
    'NO3_N+NO2_N_AVG': 'NO3-N + NO2-N Avg (mg/L)',
    'P_TOTAL_AVG': 'P-Total Avg (mg/L)',
    'P_SOLID_AVG': 'P-Solid Avg (mg/L)',
    'Electricity_consumed_onsite_kWh_per_m3': 'Electricity consumed onsite (kWh/m³)',
    'Electricity_consumed_onsite_MJ_per_m3' : 'Electricity consumed onsite (MJ/m³)',
    'Electricity_purchased_from_utility_kWh_per_m3': 'Electricity purchased from utility (kWh/m³)',
    'Electricity_produced_onsite_MJ_per_m3': 'Electricity produced onsite (MJ/m³)',
    'Electricity_produced_onsite_kWh_per_m3': 'Electricity produced onsite (kWh/m³)',
    'Electricity_produced_onsite_MJ_per_m3': 'Electricity produced onsite (MJ/m3)',
    'NATURAL_GAS_PURCHASED_onsite_MMBtu_per_m3': 'Natural Gas Purchased onsite (MMBtu/m³)',
    'NATURAL_GAS_PURCHASED_onsite_MJ_per_m3': 'Natural Gas Purchased onsite (MJ/m³)',
    'Biogas_generated_onsite_MMBtu_per_m3': 'Biogas generated onsite (MMBtu/m³)',
    'Biogas_generated_onsite_MJ_per_m3': 'Biogas generated onsite (MJ/m³)',
    'Electricity_purchased_from_utility_MJ_per_m3': 'Electricity purchased from utility (MJ/m³)',
    'Total energy purchased_MJ_per_m3_survey': 'Total energy purchased (MJ/m³ - survey)',
    'TN_AVG_to_total_nitrogen_reduction': 'TN Avg to total nitrogen reduction (mg/L)',
    'NH3_N_AVG_to_ammonia_reduction': 'NH3-N Avg to ammonia reduction (mg/L)',
    'P_TOTAL_AVG_to_total_phosphorus_reduction': 'P-Total Avg to total phosphorus reduction (mg/L)',
}

# DECIMAL PLACES DICTIONARY
decimal_places = {
    'DESIGN_FLOW': 0,
    'ACTUAL_FLOW_AVG': 0,
    'BOD5_AVG': 1,
    'CBOD5_AVG': 1,
    'SS_AVG': 1,
    'VSS_AVG': 1,
    'TS_AVG': 1,
    'TSS_AVG': 1,
    'VTS_AVG': 1,
    'TKN_AVG': 1,
    'TN_AVG': 1,
    'NH3_N_AVG': 1,
    'NO3_N_AVG': 1,
    'NO2_N_AVG': 1,
    'NO3_N+NO2_N_AVG': 1,
    'P_TOTAL_AVG': 1,
    'P_SOLID_AVG': 1,
    'Electricity_consumed_onsite_kWh_per_m3': 2,
    'Electricity_consumed_onsite_MJ_per_m3' : 2,
    'Electricity_purchased_from_utility_kWh_per_m3': 2,
    'Electricity_produced_onsite_kWh_per_m3': 2,
    'NATURAL_GAS_PURCHASED_onsite_MMBtu_per_m3': 4,
    'NATURAL_GAS_PURCHASED_onsite_MJ_per_m3': 2,
    'Biogas_generated_onsite_MMBtu_per_m3': 4,
    'Biogas_generated_onsite_MJ_per_m3': 2,
    'Electricity_purchased_from_utility_MJ_per_m3': 2,
    'Total energy purchased_MJ_per_m3_survey': 2,
    'TN_AVG_to_total_nitrogen_reduction': 1,
    'NH3_N_AVG_to_ammonia_reduction': 1,
    'P_TOTAL_AVG_to_total_phosphorus_reduction': 1,
}

# Define column-specific vertical lines
vlines_config = {
    'BOD5_AVG': {
        'names': ['EPA Standard', 'Warning Threshold'],
        'values': [10.5, 8.2],
        'colors': ['red', 'orange']
    },
    'TN_AVG': {
        'names': ['Nitrogen Limit'],
        'values': [15.0],
        'colors': ['darkred']
    },
    'P_TOTAL_AVG': {
        'names': ['Phosphorus Limit', 'Alert Level'],
        'values': [2.0, 1.5],
        'colors': ['purple', 'magenta']
    },
    'ACTUAL_FLOW_AVG': {
        'names': ['Design Capacity'],
        'values': [100.0],
        'colors': ['blue']
    }
}

# Convert columns to numeric
columns_to_convert = list(entity_to_label.keys())
df_merged_criteria[columns_to_convert] = df_merged_criteria[columns_to_convert].apply(pd.to_numeric, errors='coerce')

# Create a label with county, city, state, CWNS_ID for x-axis
df_merged_criteria['label_for_plot'] = df_merged_criteria.apply(
    lambda row: ', '.join([
        str(val) if col != 'CWNS_ID' else f'[{val}]'
        for col, val in [('CITY', row['CITY']), ('COUNTY_NAME', row['COUNTY_NAME']), 
                         ('STATE_CODE', row['STATE_CODE']), ('CWNS_ID', row['CWNS_ID'])]
        if pd.notna(val) and str(val).strip() != ''
    ]), axis=1
)

# EXAMPLE 1: Auto-calculated offset (default behavior)
create_versatile_plots(df_merged_criteria, 
                       columns_to_plot=list(entity_to_label.keys()),
                       y_axis_labels=list(entity_to_label.values()),
                       x_axis_column='label_for_plot',
                       x_axis_label='',
                       plot_title='',
                       include_statistics=False,
                       plot_type='column', 
                       single_color='steelblue', 
                       display_in_notebook=False,
                       decimal_places=decimal_places,
                       save_files=True,
                       horizontal=True,
                       output_dir=os.path.join(path_out_plots, 'column_grouped_auto_offset'),
                       group_by_column='STATE_CODE',
                       group_sort_order='asc',
                       group_label_text='State',
                       vlines_config=vlines_config,
                       font_scale=1.3,
                       bold_axis_labels=False,
                       bold_tick_labels=False,
                       bold_title=False,
                       bold_group_labels=False,
                       bold_value_labels=False
                       # value_label_offset is None - auto-calculated
                       )

# EXAMPLE 2: Custom offset (larger offset)
create_versatile_plots(df_merged_criteria, 
                       columns_to_plot=list(entity_to_label.keys()),
                       y_axis_labels=list(entity_to_label.values()),
                       x_axis_column='label_for_plot',
                       x_axis_label='',
                       plot_title='',
                       include_statistics=False,
                       plot_type='column', 
                       single_color='steelblue', 
                       display_in_notebook=False,
                       decimal_places=decimal_places,
                       save_files=True,
                       horizontal=True,
                       output_dir=os.path.join(path_out_plots, 'column_grouped_custom_offset'),
                       group_by_column='STATE_CODE',
                       group_sort_order='asc',
                       group_label_text='State',
                       vlines_config=vlines_config,
                       font_scale=1.3,
                       bold_axis_labels=False,
                       bold_tick_labels=False,
                       bold_title=False,
                       bold_group_labels=False,
                       bold_value_labels=False,
                       # Custom offset - adjust this value as needed
                       value_label_offset=5.0
                       )

# EXAMPLE 3: No offset (original behavior)
create_versatile_plots(df_merged_criteria, 
                       columns_to_plot=list(entity_to_label.keys()),
                       y_axis_labels=list(entity_to_label.values()),
                       x_axis_column='label_for_plot',
                       x_axis_label='',
                       plot_title='',
                       include_statistics=False,
                       plot_type='column', 
                       single_color='steelblue', 
                       display_in_notebook=False,
                       decimal_places=decimal_places,
                       save_files=True,
                       horizontal=True,
                       output_dir=os.path.join(path_out_plots, 'column_grouped_no_offset'),
                       group_by_column='STATE_CODE',
                       group_sort_order='asc',
                       group_label_text='State',
                       vlines_config=vlines_config,
                       font_scale=1.3,
                       bold_axis_labels=False,
                       bold_tick_labels=False,
                       bold_title=False,
                       bold_group_labels=False,
                       bold_value_labels=False,
                       # No offset
                       value_label_offset=0.0
                       )

In [ ]:
# (TEST)

import matplotlib.pyplot as plt
import pandas as pd
import os

def plot_horizontal_bars(dataframe, x_column, y_column, title="", 
                         xlabel="", ylabel="", decimals=2, 
                         bar_color="steelblue", label_format=None,
                         save_path=None):
    """
    Create a horizontal bar plot from a DataFrame with customizable features.
    
    Parameters:
    -----------
    dataframe : pd.DataFrame
        The input DataFrame containing the data to plot
    x_column : str
        Column name for the values (x-axis)
    y_column : str
        Column name for the categories (y-axis)
    title : str, optional
        Title of the plot (default: "")
    xlabel : str, optional
        Label for the x-axis (default: "")
    ylabel : str, optional
        Label for the y-axis (default: "")
    decimals : int, optional
        Number of decimal places to display on bar labels (default: 2)
    bar_color : str, optional
        Color of the bars (default: "steelblue")
    label_format : str, optional
        Custom format string for labels, e.g., "{:.2f}%" (default: None)
    save_path : str, optional
        Directory path to save the PNG file (default: None, no save)
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    
    # Create figure and axes
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Create horizontal bar plot
    bars = ax.barh(dataframe[y_column], dataframe[x_column], color=bar_color)
    
    # Add value labels on the bars
    for i, (bar, value) in enumerate(zip(bars, dataframe[x_column])):
        if label_format:
            # Use custom format if provided
            label_text = label_format.format(value)
        else:
            # Use default format with specified decimal places
            label_text = f"{value:.{decimals}f}"
        
        # Position label at the end of each bar
        ax.text(value, bar.get_y() + bar.get_height()/2, 
                f"  {label_text}", 
                va='center', ha='left', fontsize=10)
    
    # Set labels and title
    ax.set_xlabel(xlabel, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    # Adjust layout to prevent label cutoff
    plt.tight_layout()
    
    # Save figure if save_path is provided
    if save_path:
        # Create directory if it doesn't exist
        if not os.path.exists(save_path):
            os.makedirs(save_path)
        
        # Generate filename: column_name_horizontal_bar.png
        filename = f"{x_column}_horizontal_bar.png"
        filepath = os.path.join(save_path, filename)
        
        # Save the figure
        fig.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"Figure saved to: {filepath}")
    
    return fig, ax


# Example usage:
if __name__ == "__main__":
    # Create sample DataFrame
    sample_data = pd.DataFrame({
        'Product': ['Product A', 'Product B', 'Product C', 'Product D'],
        'Sales': [1234.567, 2345.891, 1876.234, 3456.789]
    })
    
    # Plot with default settings and save
    fig, ax = plot_horizontal_bars(
        dataframe=df_merged_criteria,
        x_column='ACTUAL_FLOW_AVG',
        y_column='Product',
        title='Product Sales Performance',
        xlabel='Sales ($)',
        ylabel='Product',
        decimals=2,
        bar_color='#2E86AB',
        save_path='./plots'  # Specify directory to save
    )
    
    plt.show()
    
    # Plot with custom format and save to different location
    fig2, ax2 = plot_horizontal_bars(
        dataframe=df_merged_criteria,
        x_column='Sales',
        y_column='Product',
        title='Product Sales with Percentage',
        xlabel='Sales',
        ylabel='Product',
        bar_color='#A23B72',
        label_format="{:.1f}%",
        save_path='./output/charts'  # Creates nested directories if needed
    )
    
    plt.show()

In [ ]:
# Merge with Abbaddi et al SI_C
# concat 'SI_C to column names in df_SI_C
df_sub = df_SI_C.add_suffix('_SI_C')
df_criteria_SI_C = df_merged_criteria.merge(df_sub, left_on='CWNS_ID', right_on = 'CWNS code_SI_C', how='left', suffixes=('', '_SI_C')).reset_index(drop=True)
df_criteria_SI_C.to_excel(os.path.join(path_out, 'survey_data_with_criteria_SI_C.xlsx'), index=False)

# Merge energy demand data from Abbaddi et al
# concat 'energy' to column names in df_energy_demand
df_sub = df_energy.add_suffix('_energy')
df_criteria_SI_C_energy = df_criteria_SI_C.merge(df_sub[['CWNS code_energy', 
                                                         'total electricity median [MJ·MGD-1]_energy', 
                                                         'chemical electricity [MJ·MGD-1]_energy', 
                                                         'total natural gas [MJ·MGD-1]_energy', 
                                                         'chemical natural gas [MJ·MGD-1]_energy']], 
                                                 left_on='CWNS_ID', right_on = 'CWNS code_energy', how='left', suffixes=('', '_energy')).reset_index(drop=True)

# Calclate onsite-electricity and onsite-natural gas per m3
df_criteria_SI_C_energy['onsite_electricity_MJ_per_MGD_energy'] = np.where(
    ~df_criteria_SI_C_energy['total electricity median [MJ·MGD-1]_energy'].isna() &
    ~df_criteria_SI_C_energy['chemical electricity [MJ·MGD-1]_energy'].isna(),
    df_criteria_SI_C_energy['total electricity median [MJ·MGD-1]_energy'] - df_criteria_SI_C_energy['chemical electricity [MJ·MGD-1]_energy'],
    np.nan
)
df_criteria_SI_C_energy['onsite_electricity_MJ_per_m3_energy'] = np.where(
    ~df_criteria_SI_C_energy['onsite_electricity_MJ_per_MGD_energy'].isna(),
    df_criteria_SI_C_energy['onsite_electricity_MJ_per_MGD_energy'] / 3785.41,
    np.nan
)
df_criteria_SI_C_energy['onsite_natural_gas_MJ_per_MGD_energy'] = np.where(
    ~df_criteria_SI_C_energy['total natural gas [MJ·MGD-1]_energy'].isna() &
    ~df_criteria_SI_C_energy['chemical natural gas [MJ·MGD-1]_energy'].isna(),
    df_criteria_SI_C_energy['total natural gas [MJ·MGD-1]_energy'] - df_criteria_SI_C_energy['chemical natural gas [MJ·MGD-1]_energy'],
    np.nan
)
df_criteria_SI_C_energy['onsite_natural_gas_MJ_per_m3_energy'] = np.where(
    ~df_criteria_SI_C_energy['onsite_natural_gas_MJ_per_MGD_energy'].isna(),
    df_criteria_SI_C_energy['onsite_natural_gas_MJ_per_MGD_energy'] / 3785.41,
    np.nan
)
# Total energy input
df_criteria_SI_C_energy['onsite_energy demand_MJ_per_m3_energy'] =\
df_criteria_SI_C_energy['onsite_natural_gas_MJ_per_m3_energy'] +\
df_criteria_SI_C_energy['onsite_electricity_MJ_per_m3_energy']

df_criteria_SI_C_energy.to_excel(os.path.join(path_out, 'survey_data_with_criteria_SIenergy.xlsx'), index=False)


In [ ]:
# Create histogram and capture the counts and bin edges
plt.figure(figsize=(8, 5))
counts, bins, patches = plt.hist(df_plot['FLOW_2022_MGD_FINAL'], bins=10, color='skyblue', edgecolor='black')

# Add labels above bars
for count, bin_left, bin_right in zip(counts, bins[:-1], bins[1:]):
    bin_center = (bin_left + bin_right) / 2
    plt.text(bin_center, count + 0.5, f'{int(count)}', ha='center', va='bottom', fontsize=9)

# Labels and styling
plt.xlabel('Flow in 2022 (MGD)')
plt.ylabel('Number of Facilities')
plt.title('Distribution of FLOW (MGD)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# If I select the top 50 by flow from all WWTP data how the distribution looks like?
df_top50 = df_all_plants.query("FACILITY_TYPE == 'Treatment Plant' and FLOW_2022_MGD_FINAL > 0") \
                        .nlargest(50, 'FLOW_2022_MGD_FINAL')

plt.figure(figsize=(8, 5))
counts, bins, patches = plt.hist(df_top50['FLOW_2022_MGD_FINAL'], bins=10, color='skyblue', edgecolor='black')

# Add labels above bars
for count, bin_left, bin_right in zip(counts, bins[:-1], bins[1:]):
    bin_center = (bin_left + bin_right) / 2
    plt.text(bin_center, count + 0.5, f'{int(count)}', ha='center', va='bottom', fontsize=9)

# Labels and styling
plt.xlabel('Flow in 2022 (MGD)')
plt.ylabel('Number of Facilities')
plt.title('From All plants sheet - distribution of FLOW (MGD)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# merge facility level TT assignments per Abbadi et al to top 50 selected WWTPs

df_sel_TT = pd.merge(df_select_plants, df_SI_C, left_on='CWNS_ID', right_on='CWNS code').reset_index(drop=True)
df_sel_TT.head()

In [ ]:
# Assign TT assignments to different columns
TT_expand =  pd.DataFrame ( df_sel_TT['treatment train'].str.strip("[]").str.replace("'", "").str.split(', ').tolist() ).fillna('')
TT_expand.columns = [f'TT_{i+1}' for i in TT_expand.columns]
df_sel_TT = pd.concat([df_sel_TT, TT_expand], axis=1)

df_sel_TT.head()

In [ ]:
# Summarize count of TTs

# Select TT columns and replace empty strings with NaN
TT_cols = ['TT_1', 'TT_2', 'TT_3', 'TT_4']
df_tt = df_sel_TT[TT_cols].replace('', np.nan)

# Melt into long format: one TT per row, along with its source column
df_long = df_tt.melt(var_name='TT_col', value_name='TT').dropna()

# Count how often each TT appears in each TT_X column
counts = df_long.groupby(['TT', 'TT_col']).size().unstack(fill_value=0)

# Rename columns like TT_1 → n_TT_1
counts.columns = [f'n_{col}' for col in counts.columns]

# Add total count across all TT_X columns
counts['n_TT'] = counts.sum(axis=1)

# Reset index to get 'TT' as a column
unique_TT_df = counts.reset_index()

if (write_output):
    unique_TT_df.to_csv(path_out + '/' + 'unique_TT.csv', index=False)

unique_TT_df.head()

In [ ]:
# Plot counts for TTs (all assignments combined)

# Melt the DataFrame to "long" format for Seaborn
df_melted = unique_TT_df.melt(id_vars=['TT'], value_vars=[
    #'n_TT_1', 
    #'n_TT_2', 
    #'n_TT_3', 
    #'n_TT_4', 
    'n_TT'], 
                    var_name='count_type', value_name='count')

# Create the bar plot using Seaborn
plt.figure(figsize=(10, 5))
ax = sns.barplot(x='TT', y='count', hue='count_type', data=df_melted,
            order=sorted(df_melted['TT'].unique())  # alphabetical order
            )

# Show labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge', padding=3, fontsize=9)

# Set y-ticks with step of 1, and starting from 0 to max+1
y_max = df_melted['count'].max()
plt.yticks(np.arange(0, y_max + 2, 1))  # +2 so it includes the last tick

# Adding title and labels
plt.title('Counts for Different TT Codes (all assignments combined)', fontsize=14)
plt.xlabel('TT Codes', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Plot counts for TTs (for TT Codes_1 only)

# Melt the DataFrame to "long" format for Seaborn
df_melted = unique_TT_df.melt(id_vars=['TT'], value_vars=[
    'n_TT_1', 
    #'n_TT_2', 
    #'n_TT_3', 
    #'n_TT_4', 
    #'n_TT'
    ], 
                    var_name='count_type', value_name='count')

# Create the bar plot using Seaborn
plt.figure(figsize=(10, 5))
ax = sns.barplot(x='TT', y='count', hue='count_type', data=df_melted,
            order=sorted(df_melted['TT'].unique())  # alphabetical order
            )

# Show labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge', padding=3, fontsize=9)

# Set y-ticks with step of 1, and starting from 0 to max+1
y_max = df_melted['count'].max()
plt.yticks(np.arange(0, y_max + 2, 1))  # +2 so it includes the last tick

# Adding title and labels
plt.title('Counts for Different TT Codes (primary assignments only)', fontsize=14)
plt.xlabel('TT Codes', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Summarize flow by TT

# Select TT columns and replace empty strings with NaN
TT_cols = ['TT_1', 'TT_2', 'TT_3', 'TT_4']
df_tt = df_sel_TT[TT_cols].replace('', np.nan)

# Also select the flow column
flow_col = 'FLOW_2022_MGD_FINAL'
df_tt[flow_col] = df_sel_TT[flow_col]

# Melt TT columns into long format
df_long = df_tt.melt(id_vars=[flow_col], var_name='TT_col', value_name='TT').dropna(subset=['TT'])

# Group by TT and TT_col and sum the flow
flow_sums = df_long.groupby(['TT', 'TT_col'])[flow_col].sum().unstack(fill_value=0)

# Rename columns to indicate sums
flow_sums.columns = [f'sum_{col}' for col in flow_sums.columns]

# Add total sum across all TT columns
flow_sums['sum_TT'] = flow_sums.sum(axis=1)

# Reset index to get TT as a column
unique_TT_flow_df = flow_sums.reset_index()

if write_output:
    unique_TT_flow_df.to_csv(path_out + '/' + 'unique_TT_flow.csv', index=False)

unique_TT_flow_df.head()


In [ ]:
# Plot flow for TTs (all assignments combined)

# Melt the DataFrame to "long" format for Seaborn
df_melted = unique_TT_flow_df.melt(id_vars=['TT'], value_vars=[
    #'sum_TT_1', 
    #'sum_TT_2', 
    #'sum_TT_3', 
    #'sum_TT_4', 
    'sum_TT'], 
                    var_name='sum_type', value_name='sum')

# Create the bar plot using Seaborn
plt.figure(figsize=(10, 5))
ax = sns.barplot(x='TT', y='sum', hue='sum_type', data=df_melted,
            order=sorted(df_melted['TT'].unique())  # alphabetical order
            )

# Show labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge', padding=3, fontsize=9)

# Set y-ticks with step of 1, and starting from 0 to max+1
y_max = df_melted['sum'].max()
plt.yticks(np.arange(0, y_max, 500))  

# Adding title and labels
plt.title('Sum of flow for Different TT Codes (all assignments combined)', fontsize=14)
plt.xlabel('TT Codes', fontsize=12)
plt.ylabel('Flow', fontsize=12)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Plot flow for TTs (for TT Codes_1 only)

# Melt the DataFrame to "long" format for Seaborn
df_melted = unique_TT_flow_df.melt(id_vars=['TT'], value_vars=[
    'sum_TT_1', 
    #'sum_TT_2', 
    #'sum_TT_3', 
    #'sum_TT_4', 
    #'sum_TT'
    ], 
                    var_name='sum_type', value_name='sum')

# Create the bar plot using Seaborn
plt.figure(figsize=(10, 5))
ax = sns.barplot(x='TT', y='sum', hue='sum_type', data=df_melted,
            order=sorted(df_melted['TT'].unique())  # alphabetical order
            )

# Show labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge', padding=3, fontsize=9)

# Set y-ticks with step of 1, and starting from 0 to max+1
y_max = df_melted['sum'].max()
plt.yticks(np.arange(0, y_max, 500))  

# Adding title and labels
plt.title('Sum of flow for Different TT Codes (primary assignments combined)', fontsize=14)
plt.xlabel('TT Codes', fontsize=12)
plt.ylabel('Flow', fontsize=12)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Analyze: flow rate vs. energy, flow rate vs. GHG emissions estimates

# Question: Does energy estimates and GHG emission estimates follow a linear trend w.r.t. flow rate?
# Knowing this will help us to make assumption of scale when doing GREET implementation

TT_expand =  pd.DataFrame ( df_SI_C['treatment train'].str.strip("[]").str.replace("'", "").str.split(', ').tolist() ).fillna('')
TT_expand.columns = [f'TT_{i+1}' for i in TT_expand.columns]
df_SI_C_1 = pd.concat([df_SI_C, TT_expand], axis=1)
#print(df_SI_C_1.columns)
df_SI_C_1.head()

In [ ]:
# flow rate vs onsite CH4 combustion emissions for all TTs

# sns.scatterplot(data=df_SI_C_1, x='flow [MGD]', y='CH4 [kg CO2-eq/day]')
sns.regplot(x='flow [MGD]', y='CH4 [kg CO2-eq/day]', data=df_SI_C_1, lowess=True, scatter_kws={'color': 'blue'}, line_kws={'color': 'black'})

plt.title('Flow rate vs. Onsite CH4 combustion emissions')
plt.show()

In [ ]:
# flow rate vs onsite CH4 combustion emissions by TT_1 assignment

sns.lmplot(
    data=df_SI_C_1,
    x='flow [MGD]',
    y='CH4 [kg CO2-eq/day]',
    col='TT_1',       # Facet by group
    col_wrap=4,
    lowess=True,       # Add smoother fit, change to False for linear fit
    height=4,
    aspect=1.2,
    scatter_kws={'s': 60},
    facet_kws={'sharey': False}, # separate y-axis scales
)

plt.suptitle('flow rate vs onsite CH4 combustion emissions by TT_1 assignment', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# flow rate vs onsite N2O emissions by TT_1 assignment

sns.lmplot(
    data=df_SI_C_1,
    x='flow [MGD]',
    y='N2O [kg CO2-eq/day]',
    col='TT_1',       # Facet by group
    col_wrap=4,
    lowess=True,       # Add smoother fit, change to False for linear fit
    height=4,
    aspect=1.2,
    scatter_kws={'s': 60},
    facet_kws={'sharey': False}, # separate y-axis scales
)

plt.suptitle('flow rate vs onsite N2O emissions by TT_1 assignment', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# flow rate vs onsite CO2 emissions by TT_1 assignment

sns.lmplot(
    data=df_SI_C_1,
    x='flow [MGD]',
    y='CO2 [kg CO2-eq/day]',
    col='TT_1',       # Facet by group
    col_wrap=4,
    lowess=True,       # Add smoother fit, change to False for linear fit
    height=4,
    aspect=1.2,
    scatter_kws={'s': 60},
    facet_kws={'sharey': False}, # separate y-axis scales
)

plt.suptitle('flow rate vs onsite CO2 emissions by TT_1 assignment', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# flow rate vs onsite electricity use related emissions by TT_1 assignment

sns.lmplot(
    data=df_SI_C_1,
    x='flow [MGD]',
    y='electricity [kg CO2-eq/day]',
    col='TT_1',       # Facet by group
    col_wrap=4,
    lowess=True,       # Add smoother fit, change to False for linear fit
    height=4,
    aspect=1.2,
    scatter_kws={'s': 60},
    facet_kws={'sharey': False}, # separate y-axis scales
)

plt.suptitle('flow rate vs onsite electricity use related emissions by TT_1 assignment', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# flow rate vs onsite natural gas use emissions by TT_1 assignment

sns.lmplot(
    data=df_SI_C_1,
    x='flow [MGD]',
    y='onsite natural gas [kg CO2-eq/day]',
    col='TT_1',       # Facet by group
    col_wrap=4,
    lowess=True,       # Add smoother fit, change to False for linear fit
    height=4,
    aspect=1.2,
    scatter_kws={'s': 60},
    facet_kws={'sharey': False}, # separate y-axis scales
)

plt.suptitle('flow rate vs onsite natural gas use emissions by TT_1 assignment', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# flow rate vs upstream natural gas use emissions by TT_1 assignment

sns.lmplot(
    data=df_SI_C_1,
    x='flow [MGD]',
    y='upstream natural gas [kg CO2-eq/day]',
    col='TT_1',       # Facet by group
    col_wrap=4,
    lowess=True,       # Add smoother fit, change to False for linear fit
    height=4,
    aspect=1.2,
    scatter_kws={'s': 60},
    facet_kws={'sharey': False}, # separate y-axis scales
)

plt.suptitle('flow rate vs upstream natural gas use emissions by TT_1 assignment', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# flow rate vs landfill methane emissions by TT_1 assignment

# Remove plants (with a very low cutoff) to select plants which does have landfill application

df_SI_C_2 = df_SI_C_1[ df_SI_C_1['landfill CH4 [kg CO2-eq/day]'] > 100 ] 

sns.lmplot(
    data=df_SI_C_2,
    x='flow [MGD]',
    y='landfill CH4 [kg CO2-eq/day]',
    col='TT_1',       # Facet by group
    col_wrap=4,
    lowess=False,       # Add smoother fit, change to False for linear fit
    height=4,
    aspect=1.2,
    scatter_kws={'s': 60},
    facet_kws={'sharey': False}, # separate y-axis scales
)

plt.suptitle('flow rate vs landfill methane emissions by TT_1 assignment', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# flow rate vs onsite land application N2O emissions by TT_1 assignment

# Remove plants (with a very low cutoff) to select plants which does have landfill application

df_SI_C_2 = df_SI_C_1[ df_SI_C_1['land application N2O [kg CO2-eq/day]'] > 100 ] 

sns.lmplot(
    data=df_SI_C_2,
    x='flow [MGD]',
    y='land application N2O [kg CO2-eq/day]',
    col='TT_1',       # Facet by group
    col_wrap=4,
    lowess=True,       # Add smoother fit, change to False for linear fit
    height=4,
    aspect=1.2,
    scatter_kws={'s': 60},
    facet_kws={'sharey': False}, # separate y-axis scales
)

plt.suptitle('flow rate vs onsite land application N2O emissions by TT_1 assignment', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# flow rate vs total onsite emissions by TT_1 assignment

sns.lmplot(
    data=df_SI_C_1,
    x='flow [MGD]',
    y='total onsite emission [kg CO2-eq/day]',
    col='TT_1',       # Facet by group
    col_wrap=4,
    lowess=True,       # Add smoother fit, change to False for linear fit
    height=4,
    aspect=1.2,
    scatter_kws={'s': 60},
    facet_kws={'sharey': False}, # separate y-axis scales
)

plt.suptitle('flow rate vs total onsite emissions by TT_1 assignment', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# flow rate vs total emissions by TT_1 assignment

sns.lmplot(
    data=df_SI_C_1,
    x='flow [MGD]',
    y='total emission [kg CO2-eq/day]',
    col='TT_1',       # Facet by group
    col_wrap=4,
    lowess=True,       # Add smoother fit, change to False for linear fit
    height=4,
    aspect=1.2,
    scatter_kws={'s': 60},
    facet_kws={'sharey': False}, # separate y-axis scales
)

plt.suptitle('flow rate vs total emissions by TT_1 assignment', y=1.03)
plt.tight_layout()
plt.show()